# 🧮 09 · Linear systems and matrix inverses
### Systems of equations as $C\mathbf x = \mathbf b$, the inverse of a matrix, elementary row operations and elementary matrices, Gauss–Jordan elimination, the invertible matrix theorem, Cramer's rule, under-determined systems, row-reduced echelon form, null spaces and the “−1 trick”

**BiologyToMath** · Part C — Linear algebra · Module 09 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/09_linear_systems_inverses.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy (`np.linalg`), SymPy (exact fractions, `rref`, `nullspace`), pandas, Matplotlib |
| **Prerequisites** | Modules 07–08 (vectors, dot products, matrix multiplication, transpose, identity matrix, determinant); Module 01 for Python and NumPy |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Linear systems and matrix inverses: systems of $n$ linear equations in $m$ unknowns and their matrix form $C\mathbf x = \mathbf b$; the inverse $C^{-1}$, the 2 × 2 inverse formula, singular vs nonsingular matrices; elementary row operations (Type I swap, Type II scale, Type III add a multiple of a row) and elementary matrices, their inverses and their geometric meaning; invertible ⇔ product of elementary matrices; the matrix-inversion procedure on $[A\mid I]$; algebraic properties of inverses; the invertible matrix theorem (12 equivalent conditions); Cramer's rule; Gauss–Jordan elimination on $[A\mid\mathbf b]$; inconsistent systems; under-determined systems and the parametric vector form; row-reduced echelon form; homogeneous systems, the null space and the “−1 trick” for a null-space basis |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The cell-type mixture is **SIMULATED** with the toolkit's `simulate_cell_counts` (a pseudo-bulk sample made of simulated single cells, with known true fractions). The metabolic network is a small **illustrative toy** pathway typed into a cell, not a measured network. The chemical equations (photosynthesis, glucose respiration) are standard textbook chemistry. All other matrices are small typed-in teaching examples.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Translate** a system of linear equations into the matrix form $C\mathbf x = \mathbf b$ (and back), and **draw** a system in two unknowns as lines that meet in one point, never meet, or coincide.
2. **Compute** the inverse of a 2 × 2 matrix with the formula $\frac{1}{ad-bc}\begin{bmatrix}d&-b\\-c&a\end{bmatrix}$, **verify** $CC^{-1} = C^{-1}C = I$ and **solve** $\mathbf x = C^{-1}\mathbf b$ with exact fractions.
3. **Perform** the three elementary row operations, **write** the elementary matrix of each one and its inverse, and **explain** why they never change the solution set.
4. **Invert** a matrix by Gauss–Jordan elimination on $[A\mid I]$ and **solve** a system by Gauss–Jordan elimination on $[A\mid\mathbf b]$ — by hand, step by step, and with code.
5. **Prove** $(AB)^{-1} = B^{-1}A^{-1}$ and **use** the other algebraic properties of inverses.
6. **State** the invertible matrix theorem, **test** several of its conditions on an invertible and on a singular matrix, and **apply** Cramer's rule while explaining why it is impractical for large systems.
7. **Recognise** from the row-reduced echelon form whether a system has one, none or infinitely many solutions, and **write** the solution set of an under-determined system in parametric vector form.
8. **Find** a basis of the null space with the “−1 trick”, **check** it with SymPy, and **use** null spaces to balance chemical reactions and to describe the steady states of a metabolic network.
9. **Choose** `np.linalg.solve` over `np.linalg.inv` and **justify** the choice by speed and accuracy.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the SIMULATED tissue for deconvolution | 5 min |
| 1 · Linear systems | equations ↔ $C\mathbf x=\mathbf b$; lines that meet once, never or everywhere | 15 min |
| 2 · The inverse | definition, the 2 × 2 formula, $\det = -17$, $\mathbf z = (1/17, -10/17)$, singular matrices | 20 min |
| 3 · Elementary operations | Types I–III, elementary matrices, “rotating a line around the solution”, undoing operations | 25 min |
| 4 · Inverting by Gauss–Jordan | $[C\mid I] \to [I\mid C^{-1}]$; products of elementary matrices; $(AB)^{-1} = B^{-1}A^{-1}$ | 20 min |
| 5 · Invertible matrix theorem, Cramer's rule | 12 equivalent conditions; $x_i = \det B_i/\det A$; why Cramer is slow | 20 min |
| 6 · Gauss–Jordan on $[A\mid\mathbf b]$ | solving without an inverse; $0 = 6$; row-reduced echelon form; cell-type deconvolution | 20 min |
| 7 · Under-determined systems | lines, planes and hyperplanes; a 3 × 5 system; parametric vector form | 20 min |
| 8 · Null space and the −1 trick | homogeneous systems; null-space bases; balancing photosynthesis | 15 min |
| 9 · Metabolic steady states | $S\mathbf v = \mathbf 0$ for a toy pathway; the space of steady-state fluxes | 10 min |
| 10 · `solve`, not `inv` | speed, accuracy, condition number | 5 min |
| Wrap-up | takeaways, checkpoint, discussion, lab note | 10 min |

## 🧫 The biological question

A biopsy is ground up and its RNA is sequenced. The result is one number per gene — but the tissue was a **mixture**
of T cells, B cells, macrophages and fibroblasts. *How much of each cell type was in the sample?* If we know how
strongly each marker gene is expressed in each pure cell type, every marker gives one **linear equation** in the
unknown fractions. Four markers, four cell types: four equations in four unknowns.

The same mathematics answers questions all over biology:

| Biological question | Mathematical tool | Where in this module |
|---|---|---|
| What fraction of a bulk sample is T cells, B cells, macrophages, fibroblasts? | solve a square system $S\mathbf f = \mathbf b$ | Sections 2, 6 (🌉 deconvolution) |
| What if I measure more markers than cell types and they disagree slightly? | an inconsistent, over-determined system → least squares | Section 7 (🌉) |
| How many molecules of CO₂, H₂O, glucose and O₂ make a balanced photosynthesis equation? | a homogeneous system $A\mathbf x = \mathbf 0$ and its null space | Section 8 (🌉) |
| Which combinations of reaction rates keep every metabolite at a constant level? | the null space of a stoichiometric matrix, $S\mathbf v = \mathbf 0$ | Section 9 (🌉) |
| Why can't I just divide by a matrix in my analysis script? | invertibility, the invertible matrix theorem, `solve` vs `inv` | Sections 4, 5, 10 |

Each question is answered by the same machine — **Gauss–Jordan elimination**, a systematic way of adding, scaling and
swapping equations — and by one idea: some matrices have an **inverse**, a matrix that “undoes” them, and many do not.

> 🧪 **Simulated data, with a known truth.** The bulk sample of Sections 0, 6 and 7 is built from the course's
> simulated single cells (`simulate_cell_counts`: 160 T cells, 120 B cells, 90 macrophages, 70 fibroblasts). Because
> we simulated it, we know the true cell-type fractions and the true expression level of every gene in every cell type
> — so we can check whether solving the linear system recovers the truth.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import time
from fractions import Fraction

import sympy as sp

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `gauss_jordan` reduces a matrix to row-reduced echelon form with exact fractions and can print every row operation; `format_matrix` prints a matrix (with a bar `|` before the right-hand side); `draw_graph` draws a small network; `simulate_cell_counts` makes the SIMULATED single-cell data whose cell-type marker genes are listed in `CELL_TYPE_MARKERS`. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def circle_layout(nodes: list) -> dict:
    """Place nodes evenly on a circle (first node at the top, clockwise)."""
    n = len(nodes)
    angles = np.pi / 2 - 2 * np.pi * np.arange(n) / max(n, 1)
    return {v: (float(np.cos(a)), float(np.sin(a))) for v, a in zip(nodes, angles)}


def draw_graph(edges: list, nodes=None, directed: bool = False, pos=None, ax=None, node_colors=None,
               edge_labels=None, title: str | None = None, font_size: int = 9):
    """Draw a small graph with matplotlib only (no extra packages needed).

    ``edges`` is a list of (u, v) pairs; ``nodes`` (optional) adds isolated nodes and fixes the
    order; ``pos`` maps node → (x, y) (default: ``circle_layout``).  Directed edges get arrows,
    self-loops are drawn as small rings above the node, ``edge_labels`` maps (u, v) → text (e.g. a
    weight) and ``node_colors`` maps node → colour.  Node circles grow with the length of their
    label.  Returns the Axes.
    """
    from matplotlib.transforms import offset_copy

    if nodes is None:
        nodes = []
        for u, v in edges:
            for w in (u, v):
                if w not in nodes:
                    nodes.append(w)
    pos = circle_layout(list(nodes)) if pos is None else pos
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 3.6))
    radius = {v: max(0.62 * font_size * len(str(v)) / 2 + 4, 0.9 * font_size) for v in nodes}  # points
    for u, v in edges:
        if u == v:
            ring = offset_copy(ax.transData, fig=ax.figure, x=0, y=1.25 * radius[u], units="points")
            ax.scatter([pos[u][0]], [pos[u][1]], s=(1.3 * radius[u]) ** 2, facecolors="none",
                       edgecolors="0.35", linewidths=1.3, transform=ring, zorder=1)
            continue
        style = "-|>" if directed else "-"
        ax.annotate("", xy=pos[v], xytext=pos[u],
                    arrowprops=dict(arrowstyle=style, color="0.35", lw=1.3, mutation_scale=12,
                                    shrinkA=radius[u], shrinkB=radius[v],
                                    connectionstyle="arc3,rad=0.08" if directed else "arc3,rad=0"))
        if edge_labels and (u, v) in edge_labels:
            (x1, y1), (x2, y2) = pos[u], pos[v]
            ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(edge_labels[(u, v)]), fontsize=font_size - 1,
                    ha="center", va="center", color="darkred", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    xs = [pos[v][0] for v in nodes]
    ys = [pos[v][1] for v in nodes]
    colours = [(node_colors or {}).get(v, "#cfe2f3") for v in nodes]
    ax.scatter(xs, ys, s=[(2 * radius[v]) ** 2 for v in nodes], c=colours, edgecolors="0.2",
               linewidths=1.2, zorder=2, clip_on=False)
    for v in nodes:
        ax.text(pos[v][0], pos[v][1], str(v), ha="center", va="center", fontsize=font_size, zorder=3,
                clip_on=False)
    ax.margins(0.22)
    ax.set_aspect("equal", adjustable="datalim")
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


def format_matrix(M, n_coef=None, width: int = 7) -> str:
    """Pretty-print a matrix of numbers, Fractions or SymPy rationals (complex entries with a zero
    imaginary part print as reals); a bar ``|`` is drawn after ``n_coef`` columns."""
    M = np.asarray(M, dtype=object)

    def fmt(x):
        if getattr(x, "is_Rational", False):            # a SymPy Rational/Integer → exact fraction
            x = Fraction(int(x.p), int(x.q))
        if isinstance(x, Fraction):
            return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"
        if isinstance(x, (int, np.integer)):
            return str(int(x))
        if isinstance(x, (complex, np.complexfloating)):
            if abs(x.imag) > 1e-12:
                return f"{x.real:.3g}{x.imag:+.3g}i"
            x = x.real
        x = float(x)
        return "0" if abs(x) < 1e-12 else f"{x:.4g}"

    lines = []
    for row in M:
        cells = [fmt(x).rjust(width) for x in row]
        if n_coef is not None and n_coef < len(cells):
            cells = cells[:n_coef] + ["  |"] + cells[n_coef:]
        lines.append("[" + "".join(cells) + " ]")
    return "\n".join(lines)


def gauss_jordan(M, n_coef=None, verbose: bool = False, tol: float = 1e-12):
    """Reduce a matrix to row-reduced echelon form (RREF) with the three elementary row operations.

    Integer or Fraction input is reduced *exactly* with fractions (as on paper); float input with
    floating-point arithmetic.  For an augmented matrix [A | b] pass ``n_coef`` = number of columns
    of A, so that pivots are only chosen among the coefficient columns.  With ``verbose=True`` every
    operation (Type I swap, Type II scale, Type III add a multiple of a row) is printed together
    with the matrix after it.  Returns ``(R, pivot_columns)``.
    """
    A = np.asarray(M, dtype=object)
    exact = all(isinstance(x, (int, np.integer, Fraction)) for x in A.flat)
    if exact:
        R = np.array([[Fraction(int(x)) if not isinstance(x, Fraction) else x for x in row] for row in A],
                     dtype=object)
    else:
        R = np.asarray(A, dtype=float).copy()
    n_rows, n_cols = R.shape
    n_coef = n_cols if n_coef is None else n_coef
    is_zero = (lambda x: x == 0) if exact else (lambda x: abs(x) < tol)
    pivots, r = [], 0
    for c in range(n_coef):
        if r == n_rows:
            break
        candidates = [i for i in range(r, n_rows) if not is_zero(R[i, c])]
        if not candidates:
            continue
        p = candidates[0] if exact else max(candidates, key=lambda i: abs(R[i, c]))
        if p != r:
            R[[r, p]] = R[[p, r]]
            if verbose:
                print(f"Type I:   swap R{r + 1} ↔ R{p + 1}")
                print(format_matrix(R, n_coef if n_coef < n_cols else None))
        if R[r, c] != 1:
            factor = (1 / R[r, c]) if not exact else Fraction(1) / R[r, c]
            R[r] = R[r] * factor
            if verbose:
                print(f"Type II:  R{r + 1} ← ({format_matrix([[factor]]).strip('[] ')}) · R{r + 1}")
                print(format_matrix(R, n_coef if n_coef < n_cols else None))
        for i in range(n_rows):
            if i != r and not is_zero(R[i, c]):
                beta = -R[i, c]
                R[i] = R[i] + beta * R[r]
                if verbose:
                    print(f"Type III: R{i + 1} ← R{i + 1} + ({format_matrix([[beta]]).strip('[] ')}) · R{r + 1}")
                    print(format_matrix(R, n_coef if n_coef < n_cols else None))
        if not exact:
            R[np.abs(R) < tol] = 0.0
        pivots.append(c)
        r += 1
    return R, pivots


CELL_TYPE_MARKERS = {
    "T cell": ["CD3E", "CD3D", "IL7R"],
    "B cell": ["CD79A", "MS4A1", "CD19"],
    "macrophage": ["CD68", "LYZ", "CSF1R"],
    "fibroblast": ["COL1A1", "COL1A2", "DCN"],
}


def simulate_cell_counts(n_cells=(160, 120, 90, 70), cell_types=("T cell", "B cell", "macrophage", "fibroblast"),
                         n_background: int = 28, overdispersion: float = 0.0, seed: int = 680) -> dict:
    """SIMULATED single-cell RNA counts (molecules of each gene detected in each cell).

    Each cell type has a true mean count λ for every gene: its three ``CELL_TYPE_MARKERS`` genes are
    high (λ ≈ 6–15) in that type and low (≈ 0.1–0.5) elsewhere; ``n_background`` other genes share
    similar λ (≈ 0.2–4) in all types.  Counts are Poisson(λ); with ``overdispersion`` φ > 0 each
    cell's λ is first multiplied by a Gamma(1/φ, φ) factor (mean 1), which makes counts negative-
    binomial with variance λ + φλ² — closer to real data.

    Returns a dict with ``counts`` (cells × genes, int), ``labels`` (cell type of each cell),
    ``genes``, ``rates`` (DataFrame cell types × genes of true λ) and ``source``.
    """
    rng = np.random.default_rng(seed)
    markers = [g for t in cell_types for g in CELL_TYPE_MARKERS[t]]
    genes = markers + [f"bg{i:02d}" for i in range(1, n_background + 1)]
    rates = np.zeros((len(cell_types), len(genes)))
    bg = rng.uniform(0.2, 4.0, n_background)
    for t, ctype in enumerate(cell_types):
        for j, g in enumerate(markers):
            rates[t, j] = rng.uniform(6.0, 15.0) if g in CELL_TYPE_MARKERS[ctype] else rng.uniform(0.1, 0.5)
        rates[t, len(markers):] = bg * rng.uniform(0.8, 1.25, n_background)
    labels, counts = [], []
    for t, (ctype, n) in enumerate(zip(cell_types, n_cells)):
        lam = np.tile(rates[t], (n, 1))
        if overdispersion > 0:
            lam = lam * rng.gamma(1.0 / overdispersion, overdispersion, size=lam.shape)
        counts.append(rng.poisson(lam))
        labels += [ctype] * n
    return {"counts": np.vstack(counts), "labels": np.array(labels), "genes": genes,
            "rates": pd.DataFrame(rates, index=list(cell_types), columns=genes),
            "source": "SIMULATED single-cell counts (BiologyToMath simulate_cell_counts)"}

print("🧰 toolkit loaded:", ", ".join(sorted(['CELL_TYPE_MARKERS', 'circle_layout', 'draw_graph', 'format_matrix', 'gauss_jordan', 'simulate_cell_counts'])))

In [ ]:
# ── 0.1 A SIMULATED tissue: single cells → one pseudo-bulk sample with known cell-type fractions ──
cells = simulate_cell_counts()                       # SIMULATED: 440 cells × 40 genes (Poisson counts)
counts, genes = cells["counts"], cells["genes"]
cell_types = list(cells["rates"].index)              # T cell, B cell, macrophage, fibroblast
rates = cells["rates"]                               # TRUE mean count of every gene in every cell type
f_true = pd.Series(cells["labels"]).value_counts(normalize=True)[cell_types]
bulk = counts.mean(axis=0)                           # pseudo-bulk: average molecules per cell for each gene

print(cells["source"], "→", counts.shape, "(cells × genes)")
print("true cell-type fractions in the tissue:", f_true.round(4).to_dict())
markers4 = [CELL_TYPE_MARKERS[t][0] for t in cell_types]   # one marker gene per cell type
print("one marker per cell type:", dict(zip(cell_types, markers4)))
print("pseudo-bulk level of these markers (molecules per cell):",
      {g: round(float(bulk[genes.index(g)]), 3) for g in markers4})

🔍 **What just happened?** We simulated 440 single cells — 160 T cells, 120 B cells, 90 macrophages and 70 fibroblasts —
and averaged their counts gene by gene into one **pseudo-bulk** sample, exactly what a bulk RNA measurement of the whole
tissue would see. The true fractions are $160/440 = 0.3636$, $120/440 = 0.2727$, $90/440 = 0.2045$ and
$70/440 = 0.1591$. We picked one marker gene per cell type (CD3E for T cells, CD79A for B cells, CD68 for macrophages,
COL1A1 for fibroblasts). In Section 6 we will forget the fractions and recover them from the four bulk numbers by
solving a linear system.

---
## 1 · Linear systems: many equations, one matrix equation

> 📍 **Where we are.** Module 08 taught you to *multiply* a matrix by a vector: given $C$ and $\mathbf x$, compute
> $C\mathbf x$. This module asks the reverse question — given $C$ and the result $\mathbf b$, **which $\mathbf x$ produced
> it?** That reverse question is “solving a linear system”, and it sits inside regression, deconvolution, metabolic
> modelling and almost every numerical method you will meet.

🧮 **Math refresher — solving two equations by elimination.** You solved systems like this in school:

$$3z_1 + 2z_2 = -1, \qquad z_1 - 5z_2 = 3 .$$

*In words:* we look for two numbers $z_1, z_2$ that make both equations true at the same time.
Multiply the second equation by 3 and subtract it from the first, so that $z_1$ disappears:
$(3z_1 + 2z_2) - 3(z_1 - 5z_2) = -1 - 3\cdot 3$, i.e. $3z_1 - 3z_1 + 2z_2 + 15z_2 = -10$, i.e. $17z_2 = -10$, so
$z_2 = -10/17$. Put it back into the second equation: $z_1 = 3 + 5z_2 = 3 - 50/17 = 51/17 - 50/17 = 1/17$.
Everything in this module is this school trick, organised so that it works for 3, 50 or 20,000 unknowns.

**Definition.** A **linear system** of $n$ linear equations in $m$ unknowns $x_1, \dots, x_m$ is

$$\begin{array}{ccccccccc}
C_{11}x_1 & + & C_{12}x_2 & + & \cdots & + & C_{1m}x_m & = & b_1\\
C_{21}x_1 & + & C_{22}x_2 & + & \cdots & + & C_{2m}x_m & = & b_2\\
\vdots & & \vdots & & \ddots & & \vdots & & \vdots\\
C_{n1}x_1 & + & C_{n2}x_2 & + & \cdots & + & C_{nm}x_m & = & b_n
\end{array}$$

*In words:* equation $i$ multiplies every unknown $x_j$ by a known number $C_{ij}$ (row $i$ = equation, column $j$ =
unknown), adds the results and requires the sum to equal a known number $b_i$. “Linear” means the unknowns are only
multiplied by numbers and added — no $x_1^2$, no $x_1x_2$, no $\sin x_1$.

Collect the unknowns into the vector $\mathbf x = \langle x_1, \dots, x_m\rangle\in\mathbb R^m$ (a column), the known
numbers into the **coefficient matrix** $C\in\mathbb R^{n\times m}$ and the right-hand sides into
$\mathbf b\in\mathbb R^n$ (really $\mathbb R^{n\times 1}$, a column):

$$\begin{bmatrix} C_{11} & C_{12} & \cdots & C_{1m}\\ C_{21} & C_{22} & \cdots & C_{2m}\\ \vdots & \vdots & \ddots & \vdots\\ C_{n1} & C_{n2} & \cdots & C_{nm}\end{bmatrix}
\begin{bmatrix} x_1\\ x_2\\ \vdots\\ x_m\end{bmatrix} = \begin{bmatrix} b_1\\ b_2\\ \vdots\\ b_n\end{bmatrix}
\qquad\Longleftrightarrow\qquad C\mathbf x = \mathbf b .$$

*In words:* the whole system is one matrix equation. Expanding the product with the row–column rule of Module 08
(entry $i$ of $C\mathbf x$ is row $i$ of $C$ dotted with $\mathbf x$) gives back equation $i$. Check the shapes:
$(n\times m)(m\times 1) = n\times 1$ — the unknown vector has one entry **per column** of $C$ ($m$ of them), the
right-hand side one entry **per row** ($n$ of them). For our example,

$$\begin{bmatrix} 3 & 2\\ 1 & -5\end{bmatrix}\begin{bmatrix} z_1\\ z_2\end{bmatrix} = \begin{bmatrix} -1\\ 3\end{bmatrix}.$$

*In words:* $C$ holds the four coefficients, $\mathbf z$ the two unknowns, $\mathbf b$ the two right-hand sides.

In [ ]:
# ── 1.1 From the matrix equation back to the equations (exact, with SymPy) ──
z1, z2 = sp.symbols("z1 z2")
C = sp.Matrix([[3, 2], [1, -5]])
b = sp.Matrix([-1, 3])
z = sp.Matrix([z1, z2])

lhs = C * z                                           # the matrix–vector product, symbolically
for i in range(2):
    print(f"equation {i + 1}:  {lhs[i]} = {b[i]}")
print("\nSymPy solves the system:", sp.linsolve((C, b), z1, z2))

candidate = sp.Matrix([sp.Rational(1, 17), sp.Rational(-10, 17)])
print("C @ (1/17, -10/17) =", list(C * candidate), " → equals b?", C * candidate == b)

🔍 **What just happened?** Multiplying the symbolic matrix $C$ by the symbolic vector $\mathbf z$ reproduced the two
equations $3z_1 + 2z_2 = -1$ and $z_1 - 5z_2 = 3$ — the matrix form is only a compact way of writing them. SymPy's
`linsolve` returned the single solution $\{(1/17, -10/17)\}$, the numbers we found by hand in the refresher, and
plugging them back in gives exactly $\mathbf b$. (SymPy works with exact fractions, so “equals” really means equal,
not “equal up to rounding”.)

### 1.2 The picture: two lines

An equation $C_{11}x_1 + C_{12}x_2 = b_1$ in two unknowns is a **line** in the plane: every point on it satisfies the
equation. A solution of the system is a point on **every** line at once. Two lines in the plane can meet in exactly one
point, be parallel and never meet, or be the same line. The next figure shows the three cases.

In [ ]:
# ── 1.2 Three systems of two equations in two unknowns: one, no, infinitely many solutions ──
def line_points(a1, a2, rhs, x_range=(-4, 4)):
    """Points (x, y) on the line a1·x + a2·y = rhs (vertical lines handled separately)."""
    xs = np.linspace(*x_range, 200)
    if abs(a2) > 1e-12:
        return xs, (rhs - a1 * xs) / a2
    return np.full_like(xs, rhs / a1), np.linspace(*x_range, 200)


systems = [   # (title, [(a1, a2, rhs, label), ...], solution or None, legend position)
    ("one solution", [(3, 2, -1, "3z₁ + 2z₂ = −1"), (1, -5, 3, "z₁ − 5z₂ = 3")], (1 / 17, -10 / 17), "upper right"),
    ("no solution (parallel lines)", [(-1, 3, 5, "−x₁ + 3x₂ = 5"), (1, -3, 1, "x₁ − 3x₂ = 1")], None, "lower right"),
    ("infinitely many (the same line)", [(1, 1, 2, "x + y = 2"), (2, 2, 4, "2x + 2y = 4")], None, "lower left"),
]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.7))
for ax, (title, eqs, sol, where) in zip(axes, systems):
    for (a1, a2, rhs, lab), col, ls in zip(eqs, ["tab:blue", "tab:orange"], ["-", "--"]):
        ax.plot(*line_points(a1, a2, rhs), color=col, ls=ls, lw=2, label=lab)
    if sol is not None:
        ax.plot(*sol, "ko", ms=7, zorder=5)
        ax.annotate("(1/17, −10/17)", sol, xytext=(0.6, -2.2), fontsize=8, arrowprops=dict(arrowstyle="->"))
    ax.axhline(0, color="0.8", lw=0.8); ax.axvline(0, color="0.8", lw=0.8)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal")
    ax.set_xlabel("first unknown"); ax.set_ylabel("second unknown"); ax.set_title(title)
    ax.legend(fontsize=8, loc=where)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: the running example. The two lines cross at one point, $(1/17, -10/17) \approx
(0.06, -0.59)$ — a **unique solution**. Middle: $-x_1 + 3x_2 = 5$ and $x_1 - 3x_2 = 1$ both have slope $1/3$ but
different intercepts ($5/3$ and $-1/3$); parallel lines never meet, so the system has **no solution** (it is
**inconsistent**). Right: the second equation is just twice the first, so both describe the same line and **every point
of it** is a solution. These three cases — one, none, infinitely many — are the only possibilities for *any* linear
system, of any size (Section 6 proves it with row reduction).

⚠️ **Common confusion — “as many equations as unknowns means I can solve it.”** Counting equations is not enough. The
middle and right systems both have two equations in two unknowns, yet one has no solution and the other infinitely
many. What matters is whether the equations are genuinely different (independent) and compatible. The precise
statement is the invertible matrix theorem of Section 5. As a rule of thumb:

| Shape of $C$ ($n$ equations, $m$ unknowns) | Name | Typical outcome |
|---|---|---|
| $n = m$ (square) | determined | exactly one solution — **if** $C$ is invertible; otherwise none or infinitely many |
| $n < m$ (wide) | **under-determined** | never exactly one: infinitely many solutions (or none) — Section 7 |
| $n > m$ (tall) | **over-determined** | usually no exact solution → least squares (Section 7, Module 15) |

✏️ **Your turn.** Change the second equation of the left system to $6z_1 + 4z_2 = 5$. Before running, predict which of
the three pictures you will get. What would you change to get the right-hand picture instead?

---
## 2 · The inverse of a matrix

> 📍 **Where we are.** For one equation in one unknown, $3z = 6$, we divide: $z = 6/3 = 3^{-1}\cdot 6$. Can we “divide
> by a matrix”? The answer is the **inverse** — and the reason it sometimes does not exist is the reason some biological
> measurements cannot be untangled.

The **inverse** (reciprocal) of a non-zero number $a$ is the number $b$ with $ab = 1$; the inverse of 7 is $1/7$. We use
exactly this property to define the inverse of a matrix.

**Definition.** Let $A$ be an $n\times n$ (square) matrix. $A$ is **invertible** (or **nonsingular**) if there is an
$n\times n$ matrix $B$ with

$$AB = I_n \quad\text{and}\quad BA = I_n .$$

*In words:* multiplying by $B$ on either side undoes the multiplication by $A$ and leaves the identity matrix $I_n$
(ones on the diagonal, zeros elsewhere — the matrix version of the number 1). Then $B$ is called **the inverse** of $A$
and written $A^{-1}$. If no such $B$ exists, $A$ is **singular**. The notation $A^{-1}$ is *not* “$1/A$” or an
exponent rule — it is a name for this special matrix. Two consequences are built into the definition: only **square**
matrices can have inverses (otherwise $AB$ and $BA$ could not both be $I_n$), and even a square matrix may have none.

**Solving with an inverse.** If $C$ is invertible, multiply both sides of $C\mathbf x = \mathbf b$ **on the left** by
$C^{-1}$:

$$C\mathbf x = \mathbf b \;\Longrightarrow\; C^{-1}(C\mathbf x) = C^{-1}\mathbf b \;\Longrightarrow\; (C^{-1}C)\,\mathbf x = C^{-1}\mathbf b
\;\Longrightarrow\; I\mathbf x = C^{-1}\mathbf b \;\Longrightarrow\; \mathbf x = C^{-1}\mathbf b .$$

*In words:* the inverse cancels $C$ (matrix multiplication is associative, so we may regroup the brackets), and
$I\mathbf x = \mathbf x$. Because matrix multiplication is not commutative, the side matters: $C^{-1}$ must go on the
**left** of $\mathbf b$ — “$\mathbf b\,C^{-1}$” does not even have compatible shapes. And since every step can be
reversed (multiply by $C$), $\mathbf x = C^{-1}\mathbf b$ is the **only** solution.

In [ ]:
# ── 2.1 Checking a claimed inverse: both products must give the identity ──
A_ex = np.array([[2, 1], [1, 1]])
B_ex = np.array([[1, -1], [-1, 2]])
print("A B =\n", A_ex @ B_ex)
print("B A =\n", B_ex @ A_ex)
print("both equal I₂ → B is the inverse of A:", np.array_equal(A_ex @ B_ex, np.eye(2)) and np.array_equal(B_ex @ A_ex, np.eye(2)))

🔍 **What just happened?** For $A = \begin{bmatrix}2&1\\1&1\end{bmatrix}$ and $B = \begin{bmatrix}1&-1\\-1&2\end{bmatrix}$
both products are $I_2$: for example, entry (1,1) of $AB$ is $2\cdot 1 + 1\cdot(-1) = 1$ and entry (1,2) is
$2\cdot(-1) + 1\cdot 2 = 0$. So $A$ is invertible with $A^{-1} = B$. To *verify* an inverse you only multiply; to *find*
one you need a method — first a formula for 2 × 2 matrices, then (Section 4) a method for any size.

### 2.2 The 2 × 2 inverse formula

$$C = \begin{bmatrix} a & b\\ c & d\end{bmatrix} \quad\Longrightarrow\quad
C^{-1} = \frac{1}{ad - bc}\begin{bmatrix} d & -b\\ -c & a\end{bmatrix}\qquad (\text{if } ad - bc \neq 0).$$

*In words:* swap the two diagonal entries, change the sign of the two off-diagonal entries, and divide everything by the
**determinant** $\det C = ad - bc$ (Module 08). Where does it come from? Multiply $C$ by the swapped-and-negated matrix:

$$\begin{bmatrix} a & b\\ c & d\end{bmatrix}\begin{bmatrix} d & -b\\ -c & a\end{bmatrix}
= \begin{bmatrix} ad - bc & -ab + ba\\ cd - dc & -cb + da\end{bmatrix}
= \begin{bmatrix} ad - bc & 0\\ 0 & ad - bc\end{bmatrix} = (ad - bc)\,I_2 .$$

*In words:* the product is the determinant times the identity, so dividing by the determinant gives exactly $I_2$. If
$ad - bc = 0$ there is nothing to divide by — and indeed no inverse exists.

**The running example.** $a = 3$, $b = 2$, $c = 1$, $d = -5$:

$$\det C = ad - bc = (3)(-5) - (2)(1) = -15 - 2 = -17,\qquad
C^{-1} = \frac{1}{-17}\begin{bmatrix} -5 & -2\\ -1 & 3\end{bmatrix} = \begin{bmatrix} \tfrac{5}{17} & \tfrac{2}{17}\\[2pt] \tfrac{1}{17} & -\tfrac{3}{17}\end{bmatrix}.$$

*In words:* the determinant is $-17$; dividing the swapped-and-negated matrix by $-17$ flips every sign.

⚠️ **Common confusion — the sign pattern.** You may see the determinant written as “$ad - (-b)(-c)$”, reading the
entries off the swapped-and-negated matrix. Since $(-b)(-c) = bc$ it is the same number, but the safe form to memorise is
**$ad - bc$**: main diagonal product minus off-diagonal product, computed from $C$ itself.

Check, entry by entry:

$$CC^{-1} = \begin{bmatrix} 3 & 2\\ 1 & -5\end{bmatrix}\begin{bmatrix} \tfrac{5}{17} & \tfrac{2}{17}\\[2pt] \tfrac{1}{17} & -\tfrac{3}{17}\end{bmatrix}
= \begin{bmatrix} 3\cdot\tfrac{5}{17} + 2\cdot\tfrac{1}{17} & 3\cdot\tfrac{2}{17} + 2\cdot\tfrac{-3}{17}\\[2pt]
1\cdot\tfrac{5}{17} + (-5)\cdot\tfrac{1}{17} & 1\cdot\tfrac{2}{17} + (-5)\cdot\tfrac{-3}{17}\end{bmatrix}
= \begin{bmatrix} \tfrac{17}{17} & \tfrac{0}{17}\\[2pt] \tfrac{0}{17} & \tfrac{17}{17}\end{bmatrix} = I_2 .$$

*In words:* each entry is a row of $C$ dotted with a column of $C^{-1}$; the diagonal ones give $17/17 = 1$, the others
cancel to 0. Because $C^{-1}$ exists, $C$ is **nonsingular**. The solution is then

$$\mathbf z = C^{-1}\mathbf b = \begin{bmatrix} \tfrac{5}{17} & \tfrac{2}{17}\\[2pt] \tfrac{1}{17} & -\tfrac{3}{17}\end{bmatrix}\begin{bmatrix} -1\\ 3\end{bmatrix}
= \begin{bmatrix} \tfrac{5}{17}\cdot(-1) + \tfrac{2}{17}\cdot 3\\[2pt] \tfrac{1}{17}\cdot(-1) + \tfrac{-3}{17}\cdot 3\end{bmatrix}
= \begin{bmatrix} \tfrac{1}{17}\\[2pt] -\tfrac{10}{17}\end{bmatrix}.$$

*In words:* one matrix–vector product turns the right-hand side into the solution — “solving a system is as simple as
finding a matrix inverse” (when the inverse exists).

In [ ]:
# ── 2.2 The 2 × 2 inverse by hand (exact fractions) vs NumPy ──
def inverse_2x2(M):
    """Inverse of a 2 × 2 matrix [[a, b], [c, d]] by the formula (1/(ad − bc)) [[d, −b], [−c, a]], exactly."""
    (a, b_), (c, d) = [[Fraction(v) for v in row] for row in M]
    det = a * d - b_ * c
    if det == 0:
        raise ValueError("ad − bc = 0: the matrix is singular and has no inverse")
    return np.array([[d / det, -b_ / det], [-c / det, a / det]], dtype=object), det


C_num = np.array([[3, 2], [1, -5]])
b_num = np.array([-1, 3])
C_inv, det_C = inverse_2x2(C_num)
print("det C =", det_C)
print("C⁻¹ (exact) =\n" + format_matrix(C_inv))
print("C C⁻¹ =\n" + format_matrix(C_num @ C_inv), "\nC⁻¹ C =\n" + format_matrix(C_inv @ C_num))
z_exact = C_inv @ b_num
print("z = C⁻¹ b =", [str(v) for v in z_exact])

print("\nNumPy (floating point): inv(C) =\n", np.linalg.inv(C_num))
print("5/17 =", 5 / 17, "| solve(C, b) =", np.linalg.solve(C_num, b_num), "| 1/17 =", round(1 / 17, 6))

🔍 **What just happened?** The formula gave $\det C = -17$ and $C^{-1} = \begin{bmatrix}5/17 & 2/17\\ 1/17 & -3/17\end{bmatrix}$;
multiplying in either order returns the identity, and $C^{-1}\mathbf b = (1/17, -10/17)$, exactly the hand solution.
NumPy works in floating point: its inverse shows $5/17 \approx 0.2941$, $2/17 \approx 0.1176$, $1/17 \approx 0.0588$,
$-3/17 \approx -0.1765$, and `np.linalg.solve` returns $(0.0588, -0.5882)$. Same numbers, different representation —
fractions for checking pen-and-paper work, floats for real data.

### 2.3 When there is no inverse

In [ ]:
# ── 2.3 A singular matrix: the parallel-lines system has ad − bc = 0 ──
P = np.array([[-1, 3], [1, -3]])                 # −x1 + 3x2 = 5,  x1 − 3x2 = 1
print("det P =", (-1) * (-3) - 3 * 1)
try:
    inverse_2x2(P)
except ValueError as err:
    print("our formula:", err)
try:
    np.linalg.inv(P)
except np.linalg.LinAlgError as err:
    print("np.linalg.inv refuses:", err)
print("second column = −3 × first column?", np.array_equal(P[:, 1], -3 * P[:, 0]))

🔍 **What just happened?** For the parallel-lines system of Figure 1.2, $\det = (-1)(-3) - (3)(1) = 3 - 3 = 0$: both our
formula and NumPy refuse to invert. The last line shows *why*: the second column is $-3$ times the first, so the matrix
squashes the whole plane onto a single line (its determinant — the area-scaling factor of Module 08 — is 0), and a
squashing cannot be undone. A singular matrix has no inverse, and the system $P\mathbf x = \mathbf b$ has either no
solution or infinitely many, never exactly one.

✏️ **Your turn.** Compute by hand the inverse of $\begin{bmatrix}4&7\\2&6\end{bmatrix}$ (determinant $24 - 14 = 10$),
then check with `inverse_2x2`. For which value of $k$ is $\begin{bmatrix}k&2\\3&6\end{bmatrix}$ singular?

### 📘 Lecture notes — linear systems and the inverse

**Linear system.** $n$ equations in $m$ unknowns, $\sum_{j=1}^{m} C_{ij}x_j = b_i$ for $i = 1,\dots,n$, written
$C\mathbf x = \mathbf b$ with $C\in\mathbb R^{n\times m}$, $\mathbf x\in\mathbb R^m$, $\mathbf b\in\mathbb R^n$.
Expanding the product recovers the equations. In two unknowns each equation is a line; the system has one, no, or
infinitely many solutions.

**Inverse.** For square $A\in\mathbb R^{n\times n}$, $A^{-1}$ is the matrix with $AA^{-1} = A^{-1}A = I_n$ (if it
exists, it is unique). $A$ is **nonsingular**/invertible if $A^{-1}$ exists, **singular** otherwise. Only square
matrices can be invertible. If $C$ is invertible, $C\mathbf x = \mathbf b \iff \mathbf x = C^{-1}\mathbf b$
(multiply on the **left**).

**2 × 2 formula.** $\begin{bmatrix}a&b\\c&d\end{bmatrix}^{-1} = \dfrac{1}{ad-bc}\begin{bmatrix}d&-b\\-c&a\end{bmatrix}$,
valid iff $ad - bc\neq 0$.

**Worked example.** $3z_1 + 2z_2 = -1$, $z_1 - 5z_2 = 3$: $\det C = (3)(-5) - (2)(1) = -17$,
$C^{-1} = \frac{1}{17}\begin{bmatrix}5&2\\1&-3\end{bmatrix}$, $CC^{-1} = I_2$,
$\mathbf z = C^{-1}\mathbf b = (1/17, -10/17)$. Another pair of inverses: $\begin{bmatrix}2&1\\1&1\end{bmatrix}$ and
$\begin{bmatrix}1&-1\\-1&2\end{bmatrix}$ (check $AB = BA = I_2$).

**Common confusions.** Counting equations does not decide solvability (parallel lines: 2 equations, 2 unknowns, no
solution). $C^{-1}\mathbf b \ne \mathbf b\,C^{-1}$ (the latter is not even defined). $A^{-1}$ is not the matrix of
reciprocals $1/a_{ij}$.

> 🎓 **Exam tip.** For a 2 × 2 inverse: compute $ad - bc$ first (if it is 0, stop — singular), then swap $a\leftrightarrow d$,
> negate $b$ and $c$, divide by the determinant, and **always** check one product $CC^{-1} = I$; it takes 30 seconds and
> catches most sign errors.

---
## 3 · Elementary row operations and elementary matrices

> 📍 **Where we are.** The 2 × 2 formula does not generalise nicely. The general method is the school elimination trick,
> made systematic: three kinds of **elementary row operations** that simplify a system without changing its solutions.
> Each operation is itself a matrix multiplication — which is the key to *finding* inverses (Section 4).

**The three elementary operations** on the equations (= rows of the matrix representing the system):

| Type | Operation | Notation |
|---|---|---|
| **I** (swap rows) | exchange two equations | $R_i \leftrightarrow R_j$ |
| **II** (scale a row) | multiply an equation by a constant $\lambda\in\mathbb R\setminus\{0\}$ | $R_i \leftarrow \lambda R_i$ |
| **III** (add a row to another) | add a multiple $\beta$ of one equation to another | $R_i \leftarrow R_i + \beta R_j$ ($i\ne j$) |

**Why they keep the solution set.** If $\mathbf x$ satisfies the old equations, it satisfies each new equation, because
every new equation is a sum of multiples of old ones. Conversely, every operation can be **undone** by another operation
of the same type — swap again; scale by $1/\lambda$; add $-\beta R_j$ — so a solution of the new system also solves the
old one. That is why Type II requires $\lambda\neq 0$: multiplying an equation by 0 turns it into $0 = 0$, information
is lost and cannot be recovered.

**Elementary matrices.** An $n\times n$ matrix obtained from $I_n$ by performing a **single** row operation is called an
**elementary matrix**. For 2 × 2 matrices the three types look like

$$E_1 = \begin{pmatrix}0&1\\1&0\end{pmatrix},\qquad
E_2 = \begin{pmatrix}r&0\\0&1\end{pmatrix}\text{ or }\begin{pmatrix}1&0\\0&r\end{pmatrix},\qquad
E_3 = \begin{pmatrix}1&s\\0&1\end{pmatrix}\text{ or }\begin{pmatrix}1&0\\s&1\end{pmatrix}.$$

*In words:* $E_1$ is the identity with its rows swapped (Type I), $E_2$ the identity with one row scaled by $r\neq 0$
(Type II), $E_3$ the identity with one extra entry $s$ off the diagonal (Type III).

**Proposition.** Let $A$ be $n\times p$ and $E$ an elementary $n\times n$ matrix. Then $EA$ is the matrix obtained by
performing the row operation of $E$ on $A$.

*In words:* row operations are **left multiplications** (pre-multiplications). Why: row $i$ of $EA$ is row $i$ of $E$
times $A$, i.e. a combination of the rows of $A$ with the weights in row $i$ of $E$; row $i$ of $I$ just picks row $i$ of
$A$, and a modified row of $I$ picks the modified combination.

In [ ]:
# ── 3.1 Elementary matrices act on the rows of a general 2 × 2 matrix ──
a, b_s, c, d, r, s = sp.symbols("a b c d r s")
M = sp.Matrix([[a, b_s], [c, d]])
E1 = sp.Matrix([[0, 1], [1, 0]])          # Type I: swap
E2 = sp.Matrix([[r, 0], [0, 1]])          # Type II: scale row 1 by r
E3 = sp.Matrix([[1, s], [0, 1]])          # Type III: add s·(row 2) to row 1
for name, E in [("E1 (swap)", E1), ("E2 (scale row 1 by r)", E2), ("E3 (row 1 + s·row 2)", E3)]:
    print(f"{name}:  E·[[a, b], [c, d]] = {(E * M).tolist()}")

🔍 **What just happened?** With symbols we see the proposition in full generality:
$E_1\begin{pmatrix}a&b\\c&d\end{pmatrix} = \begin{pmatrix}c&d\\a&b\end{pmatrix}$ (rows swapped),
$E_2$ gives $\begin{pmatrix}ra&rb\\c&d\end{pmatrix}$ (row 1 scaled), and $E_3$ gives
$\begin{pmatrix}a+sc&b+sd\\c&d\end{pmatrix}$ (row 1 plus $s$ times row 2). The same idea works for any size.

### 3.2 The three types on a 3 × 2 matrix

Recall from Module 08 that $I_3A = A$. Elementary operations are small manipulations of that identity. Take
$A = \begin{bmatrix}1&4\\2&5\\3&6\end{bmatrix}$ and three elementary matrices, one of each type:

$$E = \begin{bmatrix}1&0&0\\0&0&1\\0&1&0\end{bmatrix},\qquad G = \begin{bmatrix}1&0&0\\0&1&0\\0&0&\alpha\end{bmatrix}\ (\alpha\neq0),\qquad
H = \begin{bmatrix}1&0&0\\0&1&0\\\beta&0&1\end{bmatrix}\ (\beta\in\mathbb R).$$

*In words:* $E$ is $I_3$ with rows 2 and 3 swapped (Type I), $G$ is $I_3$ with row 3 multiplied by $\alpha$ (Type II),
$H$ is $I_3$ with $\beta$ placed in row 3, column 1, i.e. “add $\beta\times$ row 1 to row 3” (Type III).

In [ ]:
# ── 3.2 E (Type I), G (Type II) and H (Type III) acting on A = [[1, 4], [2, 5], [3, 6]] ──
alpha, beta = sp.symbols("alpha beta")
A32 = sp.Matrix([[1, 4], [2, 5], [3, 6]])
E = sp.Matrix([[1, 0, 0], [0, 0, 1], [0, 1, 0]])
G = sp.Matrix([[1, 0, 0], [0, 1, 0], [0, 0, alpha]])
H = sp.Matrix([[1, 0, 0], [0, 1, 0], [beta, 0, 1]])
print("I3 A =", (sp.eye(3) * A32).tolist())
print("E A  =", (E * A32).tolist(), "  ← rows 2 and 3 swapped")
print("G A  =", (G * A32).tolist(), "  ← row 3 multiplied by α")
print("H A  =", (H * A32).tolist(), "  ← β × row 1 added to row 3")

# undoing each operation: the inverse of an elementary matrix is the elementary matrix of the reverse operation
E_rev = E                                               # swap rows 2 and 3 again
G_rev = sp.Matrix([[1, 0, 0], [0, 1, 0], [0, 0, 1 / alpha]])
H_rev = sp.Matrix([[1, 0, 0], [0, 1, 0], [-beta, 0, 1]])
for name, Rv, Op in [("E", E_rev, E), ("G", G_rev, G), ("H", H_rev, H)]:
    print(f"{name}⁻¹({name}A) == A ? {sp.simplify(Rv * (Op * A32) - A32) == sp.zeros(3, 2)}",
          f"  and {name}⁻¹ = {Rv.tolist()}")

🔍 **What just happened?** $EA$ is $A$ with rows 2 and 3 swapped ($[[1,4],[3,6],[2,5]]$), exactly as $E$ is $I_3$ with
rows 2 and 3 swapped; $GA$ multiplies the last row by $\alpha$ ($[3\alpha, 6\alpha]$); and $HA$ has last row
$[3 + \beta,\ 6 + 4\beta]$ — the first row $[1, 4]$ times $\beta$ added to the bottom row. Each operation can be
**undone** by a reversing matrix of the same type: $E^{-1} = E$ (swap back), $G^{-1}$ scales row 3 by $1/\alpha$
(this is where $\alpha\ne 0$ is needed), and $H^{-1}$ has $-\beta$ in place of $\beta$ (subtract what was added). SymPy
confirms $E^{-1}(EA) = A$, $G^{-1}(GA) = A$ and $H^{-1}(HA) = A$ for every $\alpha\neq0$ and $\beta$. So **the inverse of
an elementary matrix is again an elementary matrix**.

### 3.3 What the operations do to the lines

Each operation has a picture. **Type I** only changes the order in which we list the equations — the lines, and their
intersection, stay put. **Type II** multiplies an equation by a non-zero number: $2x - y = 3$ and $4x - 2y = 6$ are the
same line (the constant “reduces out”). **Type III** is the interesting one. Take two lines with the solution $(2, 1)$:

$$L_1:\ x + y = 3 \;\;(y = -x + 3), \qquad L_2:\ 2x - y = 3 \;\;(y = 2x - 3).$$

*In words:* $(2,1)$ lies on both lines because $2 + 1 = 3$ and $2\cdot 2 - 1 = 3$.
Replace $L_2$ by $L_2' = L_2 + k\,L_1$; with $k = -1$:

$$(2x - y) + (-1)(x + y) = 3 + (-1)\cdot 3 \;\Longrightarrow\; x - 2y = 0 \;\Longrightarrow\; y = 0.5\,x .$$

*In words:* adding a multiple of one equation to another gives a **new line that still passes through $(2,1)$**, because
$(2,1)$ satisfies both original equations and hence any combination of them ($2 - 2\cdot1 = 0$ ✓). Varying $k$ swings
the new line around the solution like a door around its hinge — “in the plane you can think of it as a rotation around
the solution”.

In [ ]:
# ── 3.3 Type III operations swing a line around the solution (2, 1) ──
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
ax.plot(*line_points(1, 1, 3), color="tab:blue", lw=2, label="L1: x + y = 3")
ax.plot(*line_points(2, -1, 3), color="tab:red", lw=2, label="L2: 2x − y = 3")
ax.plot(*line_points(1, -2, 0), color="tab:green", lw=2, ls="--", label="L2′ = L2 − L1: x − 2y = 0")
ax.plot(2, 1, "ko", ms=7, zorder=5); ax.annotate("solution (2, 1)", (2, 1), xytext=(2.4, 0.2), fontsize=8)
ax.set_title("one Type III operation (k = −1)")
ax = axes[1]
ks = [-3, -2, -1, -0.5, 0.5, 1, 2, 5]
cmap = plt.get_cmap("viridis")
for i, k in enumerate(ks):                                   # L2 + k L1:  (2 + k) x + (−1 + k) y = 3 + 3k
    ax.plot(*line_points(2 + k, -1 + k, 3 + 3 * k), color=cmap(i / (len(ks) - 1)), lw=1.4, label=f"k = {k}")
ax.plot(2, 1, "ko", ms=7, zorder=5)
ax.set_title("L2 + k·L1 for several k: a fan of lines through (2, 1)")
for ax in axes:
    ax.axhline(0, color="0.8", lw=0.8); ax.axvline(0, color="0.8", lw=0.8)
    ax.set_xlim(-1, 4); ax.set_ylim(-1, 4); ax.set_aspect("equal")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: the green dashed line $x - 2y = 0$ (slope $0.5$) is the result of the Type III operation
with $k = -1$; it passes through $(2, 1)$ like the two original lines. Right: for every $k$ the combined equation
$(2 + k)x + (-1 + k)y = 3 + 3k$ is satisfied by $(2,1)$, so all the lines pivot around that point. Elimination exploits
this freedom: we choose $k$ so that the new line is **horizontal or vertical** (one unknown eliminated) — the solution
can then be read off. Here $k = 1$ gives $3x = 6$, a vertical line $x = 2$; $k = -2$ gives $-3y = -3$, the horizontal
line $y = 1$.

### 3.4 Undoing an operation: a 2 × 2 example

How do we find an inverse? Start small: $A = \begin{bmatrix}2&1\\1&1\end{bmatrix}$ and the elementary matrix
$E = \begin{bmatrix}1&0\\-1&1\end{bmatrix}$, which performs “row 2 minus row 1”.

In [ ]:
# ── 3.4 E performs "row 2 minus row 1"; its inverse performs "row 2 plus row 1" ──
A2 = np.array([[2, 1], [1, 1]])
E21 = np.array([[1, 0], [-1, 1]])
EA = E21 @ A2
E21_inv = np.array([[1, 0], [1, 1]])
print("E A =\n", EA)
print("E⁻¹ =\n", E21_inv, "\nE⁻¹ E = I ?", np.array_equal(E21_inv @ E21, np.eye(2)))
print("E⁻¹ (E A) =\n", E21_inv @ EA, "\n= A ?", np.array_equal(E21_inv @ EA, A2))

🔍 **What just happened?** $EA = \begin{bmatrix}2&1\\-1&0\end{bmatrix}$: the second row became $[1,1] - [2,1] = [-1,0]$.
The reversing matrix adds row 1 back, $E^{-1} = \begin{bmatrix}1&0\\1&1\end{bmatrix}$, and indeed $E^{-1}(EA) = A$ —
the transformation is successfully undone. Elementary operations are reversible steps; a chain of them that turns $A$
into $I$ therefore *is* an inverse of $A$, recorded one step at a time. Section 4 turns that observation into an
algorithm.

✏️ **Your turn.** Write down the 3 × 3 elementary matrix that subtracts 4 × row 1 from row 2, and its inverse. Multiply
the two (by hand or with `@`) and check that you get $I_3$.

### 📘 Lecture notes — elementary operations and elementary matrices

**The three elementary row operations** keep the solution set of a system and transform it into a simpler form:
**Type I** swap two rows ($R_i\leftrightarrow R_j$); **Type II** multiply a row by $\lambda\in\mathbb R\setminus\{0\}$
($R_i\leftarrow\lambda R_i$); **Type III** add a multiple of another row ($R_i\leftarrow R_i+\beta R_j$). Each is
reversible by an operation of the same type.

**Elementary matrix.** An $n\times n$ matrix obtained from $I_n$ by a single row operation. **Proposition:** for
$A\in\mathbb R^{n\times p}$, $EA$ = $A$ with that row operation performed (left/pre-multiplication).

**Worked example** ($A = \begin{bmatrix}1&4\\2&5\\3&6\end{bmatrix}$): $E$ = $I_3$ with rows 2, 3 swapped →
$EA = \begin{bmatrix}1&4\\3&6\\2&5\end{bmatrix}$; $G = \operatorname{diag}(1,1,\alpha)$ → last row $[3\alpha, 6\alpha]$;
$H$ = $I_3$ with $\beta$ at position (3,1) → last row $[3+\beta, 6+4\beta]$. Reversing matrices: $E^{-1} = E$,
$G^{-1} = \operatorname{diag}(1,1,1/\alpha)$, $H^{-1}$ = $I_3$ with $-\beta$ at (3,1). 2 × 2 example:
$E = \begin{bmatrix}1&0\\-1&1\end{bmatrix}$ (row 2 − row 1), $E^{-1} = \begin{bmatrix}1&0\\1&1\end{bmatrix}$.

**Geometry (two unknowns).** Type I: same lines, same intersection. Type II: same line. Type III: a new line through
the same solution — e.g. $x + y = 3$, $2x - y = 3$ (solution $(2,1)$), $L_2' = L_2 - L_1$: $x - 2y = 0$, i.e.
$y = 0.5x$, still through $(2,1)$.

**Common confusions.** Row operations multiply on the **left** ($EA$), never on the right ($AE$ acts on columns).
Scaling by 0 is not an elementary operation.

> 🎓 **Exam tip.** To write the elementary matrix of an operation, do the operation **to $I_n$**. To invert it, write the
> elementary matrix of the reverse operation (swap back, divide by $\lambda$, subtract $\beta$).

---
## 4 · Finding an inverse: Gauss–Jordan elimination on $[A\mid I]$

> 📍 **Where we are.** Elementary matrices are invertible and they perform row operations. Put the two facts together and
> an algorithm for the inverse of *any* square matrix falls out.

Two facts (their full proofs are beyond this course; see Hoffman & Kunze in the references):

**Theorem 1.** $A\in\mathbb R^{n\times n}$ is invertible (nonsingular) **if and only if** there is a set of elementary
matrices $\{E_1, E_2, \dots, E_k\}$ with

$$A = E_1E_2\cdots E_k .$$

*In words:* invertible matrices are exactly the matrices you can build by multiplying elementary matrices together.

**Theorem 2.** If $B_1, B_2, \dots, B_k\in\mathbb R^{n\times n}$ are all invertible, their product is invertible and

$$(B_1B_2\cdots B_{k-1}B_k)^{-1} = B_k^{-1}B_{k-1}^{-1}\cdots B_2^{-1}B_1^{-1}.$$

*In words:* to undo a chain of operations, undo them in **reverse order** — like taking off shoes before socks.

**The algorithm.** Suppose row operations with elementary matrices $E_1$ (first), $E_2$ (second), …, $E_k$ (last) turn
$A$ into $I_n$. Then

$$E_k\cdots E_2E_1\,A = I_n \quad\Longrightarrow\quad E_k\cdots E_2E_1 = A^{-1}.$$

*In words:* the product of the operations, multiplied with $A$, gives $I_n$ — so by the definition of the inverse, that
product **is** $A^{-1}$. (Multiply both sides of the left equation on the right by $A^{-1}$.) And from Theorem 2,
$A = (A^{-1})^{-1} = E_1^{-1}E_2^{-1}\cdots E_k^{-1}$, a product of elementary matrices, as Theorem 1 promised. How do we
collect the product $E_k\cdots E_1$ without multiplying matrices? Apply the same operations to $I_n$ written next to $A$:

$$E_k\cdots E_1\,[\,A \mid I_n\,] = [\,E_k\cdots E_1A \mid E_k\cdots E_1I_n\,] = [\,I_n \mid A^{-1}\,].$$

*In words:* row operations act on whole rows, so they act on both halves of the **partitioned matrix** at once; when the
left half has become $I_n$, the right half has recorded $A^{-1}$.

**Matrix inversion procedure.**
1. Create the partitioned matrix $[A\mid I_n]$.
2. Apply a row operation to it.
3. While the left-hand side is not $I_n$, repeat step 2. (If a whole row of the left half becomes 0, stop: $A$ is
   singular.)

The next cell does it with exact fractions, for $C = \begin{bmatrix}3&2\\1&-5\end{bmatrix}$, using the same four
operations you would choose on paper: subtract 3 × row 2 from row 1 (Type III, $\beta = -3$); divide row 1 by 17
(Type II, $\alpha = 1/17$); swap the rows (Type I); add 5 × row 2 to row 1 (Type III, $\beta = 5$).

In [ ]:
# ── 4.1 A hand-guided row reducer: apply named elementary operations exactly and print every step ──
def apply_row_ops(M, steps, n_coef=None, verbose=True):
    """Apply elementary row operations to M with exact fractions, printing the matrix after each one.

    Each step is ("I", i, j)  → swap rows i and j;  ("II", i, alpha) → R_i ← alpha·R_i;
    ("III", i, j, beta) → R_i ← R_i + beta·R_j.  Rows are numbered from 1, as on paper; alpha and beta may be
    ints, Fractions or strings such as "1/17".  Returns the final matrix and the elementary matrix of every step.
    """
    R = np.array([[Fraction(v) for v in row] for row in M], dtype=object)
    n = R.shape[0]
    used = []
    for step in steps:
        E_step = np.array([[Fraction(int(p == q)) for q in range(n)] for p in range(n)], dtype=object)
        if step[0] == "I":
            i, j = step[1] - 1, step[2] - 1
            R[[i, j]] = R[[j, i]]
            E_step[[i, j]] = E_step[[j, i]]
            label = f"Type I:   swap R{i + 1} ↔ R{j + 1}"
        elif step[0] == "II":
            i, factor = step[1] - 1, Fraction(step[2])
            if factor == 0:
                raise ValueError("Type II needs a non-zero factor")
            R[i] = R[i] * factor
            E_step[i, i] = factor
            label = f"Type II:  R{i + 1} ← ({factor}) · R{i + 1}"
        else:
            i, j, factor = step[1] - 1, step[2] - 1, Fraction(step[3])
            R[i] = R[i] + factor * R[j]
            E_step[i, j] = factor
            label = f"Type III: R{i + 1} ← R{i + 1} + ({factor}) · R{j + 1}"
        used.append(E_step)
        if verbose:
            print(label)
            print(format_matrix(R, n_coef))
    return R, used


C_I = np.hstack([C_num, np.eye(2, dtype=int)])          # the partitioned matrix [C | I]
print("[C | I] =\n" + format_matrix(C_I, 2))
steps_inverse = [("III", 1, 2, -3), ("II", 1, "1/17"), ("I", 1, 2), ("III", 1, 2, 5)]
R_final, elem = apply_row_ops(C_I, steps_inverse, n_coef=2)
print("\nright half = C⁻¹ from Section 2?", np.array_equal(R_final[:, 2:], C_inv))

🔍 **What just happened?** Step by step, exactly as on paper:

* $R_1 \leftarrow R_1 - 3R_2$: $[3, 2 \mid 1, 0] - 3\,[1, -5 \mid 0, 1] = [0, 17 \mid 1, -3]$;
* $R_1 \leftarrow \tfrac{1}{17}R_1$: $[0, 1 \mid \tfrac{1}{17}, -\tfrac{3}{17}]$;
* swap $R_1 \leftrightarrow R_2$: the leading 1 of the first column moves to the top;
* $R_1 \leftarrow R_1 + 5R_2$: $[1, -5 \mid 0, 1] + 5\,[0, 1 \mid \tfrac1{17}, -\tfrac3{17}] = [1, 0 \mid \tfrac5{17}, \tfrac2{17}]$.

The left half is now $I_2$ and the right half is $\begin{bmatrix}5/17 & 2/17\\ 1/17 & -3/17\end{bmatrix}$ — exactly the
inverse the 2 × 2 formula gave. This is **not the only way**: any sequence of operations that reaches $I_2$ gives the
same right half, because the inverse is unique. The next cell lets the toolkit's `gauss_jordan` choose its own
operations, and multiplies out the four elementary matrices we just used.

In [ ]:
# ── 4.2 Same inverse by another route, and the inverse as a product of elementary matrices ──
R_gj, pivots = gauss_jordan(C_I, n_coef=2, verbose=True)      # the toolkit's own choice of operations
print("gauss_jordan's right half equals ours?", np.array_equal(R_gj[:, 2:], C_inv))

E_1, E_2, E_3, E_4 = elem                                       # the four elementary matrices of Section 4.1
product = E_4 @ E_3 @ E_2 @ E_1
print("\nE4 E3 E2 E1 =\n" + format_matrix(product))
print("equals C⁻¹?", np.array_equal(product, C_inv))
inv_elem = [apply_row_ops(np.eye(2, dtype=int), [s], verbose=False)[0] for s in
            [("III", 1, 2, 3), ("II", 1, 17), ("I", 1, 2), ("III", 1, 2, -5)]]   # the reverse operations
rebuilt = inv_elem[0] @ inv_elem[1] @ inv_elem[2] @ inv_elem[3]  # E1⁻¹ E2⁻¹ E3⁻¹ E4⁻¹
print("E1⁻¹ E2⁻¹ E3⁻¹ E4⁻¹ =\n" + format_matrix(rebuilt), "\n= C ?", np.array_equal(rebuilt, C_num))

🔍 **What just happened?** `gauss_jordan` took a different path — it first scaled row 1 by $1/3$, then eliminated, and
met fractions like $-17/3$ on the way — but it ended at the same right half, the same $C^{-1}$. Multiplying our four
elementary matrices in the order **last to first**, $E_4E_3E_2E_1$, reproduced $C^{-1}$, as the algorithm promised; and
the product of their inverses in the order **first to last**, $E_1^{-1}E_2^{-1}E_3^{-1}E_4^{-1}$, rebuilt $C$ itself —
$C$ is a product of four elementary matrices, the witness that Theorem 1 asks for.

### 4.3 Algebraic properties of inverses

Let $A$ and $B$ be invertible $n\times n$ matrices.

**(i) $(A^{-1})^{-1} = A$.** The definition $AA^{-1} = A^{-1}A = I$ is symmetric in $A$ and $A^{-1}$: it says just as
well that $A$ is the inverse of $A^{-1}$.

**(ii) $(AB)^{-1} = B^{-1}A^{-1}$ (note the order).** Proof — check that $B^{-1}A^{-1}$ satisfies the definition:

$$(AB)(B^{-1}A^{-1}) = A\,(BB^{-1})\,A^{-1} = A\,I_n\,A^{-1} = AA^{-1} = I_n,$$
$$(B^{-1}A^{-1})(AB) = B^{-1}\,(A^{-1}A)\,B = B^{-1}\,I_n\,B = B^{-1}B = I_n .$$

*In words:* regroup the brackets (associativity), let the innermost pair cancel, and the next pair cancels too. The
inverse of a product is the product of the inverses **in reverse order**.

**Why not $A^{-1}B^{-1}$?** If it were the inverse, we would have $I_n = (AB)(A^{-1}B^{-1}) = ABA^{-1}B^{-1}$. If $A$ and
$B$ commute ($AB = BA$) this works: $ABA^{-1}B^{-1} = BAA^{-1}B^{-1} = B(AA^{-1})B^{-1} = BB^{-1} = I_n$. In general they
do not, so: $(AB)^{-1} = A^{-1}B^{-1}$ **if and only if** $AB = BA$.

*In words:* swapping the order is only harmless when the two matrices commute.

**(iii) $(A^\top)^{-1} = (A^{-1})^\top$.** Using $(XY)^\top = Y^\top X^\top$ from Module 08:
$A^\top(A^{-1})^\top = (A^{-1}A)^\top = I^\top = I$ (and the same for the other order).

**(iv) $\det(A^{-1}) = 1/\det(A)$.** Using $\det(XY) = \det X\det Y$ from Module 08: $\det(A)\det(A^{-1}) = \det(AA^{-1})
= \det I = 1$. In particular an invertible matrix has $\det A \neq 0$.

In [ ]:
# ── 4.3 Checking the four properties exactly with SymPy ──
A_s = sp.Matrix([[2, 1], [1, 1]])
B_s = sp.Matrix([[3, 2], [1, -5]])                       # = C
print("(A⁻¹)⁻¹ == A           :", (A_s.inv()).inv() == A_s)
print("(AB)⁻¹ == B⁻¹A⁻¹       :", (A_s * B_s).inv() == B_s.inv() * A_s.inv())
print("(AB)⁻¹ == A⁻¹B⁻¹       :", (A_s * B_s).inv() == A_s.inv() * B_s.inv(), "  (AB == BA?", A_s * B_s == B_s * A_s, ")")
B_comm = A_s ** 2 + 3 * A_s                              # a matrix that commutes with A
print("with a commuting B     :", (A_s * B_comm).inv() == A_s.inv() * B_comm.inv(), "  (AB == BA?", A_s * B_comm == B_comm * A_s, ")")
print("(Aᵀ)⁻¹ == (A⁻¹)ᵀ       :", (B_s.T).inv() == (B_s.inv()).T)
print("det(C⁻¹) =", B_s.inv().det(), " and 1/det(C) =", 1 / B_s.det())

🔍 **What just happened?** All the properties hold exactly. For $A = \begin{bmatrix}2&1\\1&1\end{bmatrix}$ and $C$,
$(AC)^{-1} = C^{-1}A^{-1}$ is `True` while $A^{-1}C^{-1}$ is `False` — and so is $AC = CA$. With a matrix that does commute
with $A$ (any polynomial in $A$, here $A^2 + 3A$), the order no longer matters. Finally $\det(C^{-1}) = -1/17 = 1/\det C$.

✏️ **Your turn.** Prove $(ABC)^{-1} = C^{-1}B^{-1}A^{-1}$ for invertible $A, B, C$ by applying property (ii) twice. Then
check it with SymPy for three 2 × 2 matrices of your choice.

### 📘 Lecture notes — inverting a matrix by Gauss–Jordan elimination

**Facts.** $A\in\mathbb R^{n\times n}$ is invertible **iff** $A = E_1E_2\cdots E_k$ for some elementary matrices. A
product of invertible matrices is invertible, with $(B_1\cdots B_k)^{-1} = B_k^{-1}\cdots B_1^{-1}$. If the operations
$E_1, \dots, E_k$ (in this order) turn $A$ into $I_n$, then $(E_k\cdots E_1)A = I_n$, so $A^{-1} = E_k\cdots E_1$:
**the operations that reduce $A$ to $I$ turn $I$ into $A^{-1}$.**

**Procedure.** (1) form $[A\mid I_n]$; (2) apply a row operation; (3) repeat until the left side is $I_n$; read $A^{-1}$
on the right. A zero row on the left means $A$ is singular.

**Worked example.** $\left[\begin{array}{cc|cc}3&2&1&0\\1&-5&0&1\end{array}\right]$
$\xrightarrow{R_1 - 3R_2}$ $\left[\begin{array}{cc|cc}0&17&1&-3\\1&-5&0&1\end{array}\right]$
$\xrightarrow{R_1/17}$ $\left[\begin{array}{cc|cc}0&1&\frac1{17}&-\frac3{17}\\1&-5&0&1\end{array}\right]$
$\xrightarrow{R_1\leftrightarrow R_2}$ $\left[\begin{array}{cc|cc}1&-5&0&1\\0&1&\frac1{17}&-\frac3{17}\end{array}\right]$
$\xrightarrow{R_1 + 5R_2}$ $\left[\begin{array}{cc|cc}1&0&\frac5{17}&\frac2{17}\\0&1&\frac1{17}&-\frac3{17}\end{array}\right]$
($\beta = -3$, $\alpha = 1/17$, Type I, $\beta = 5$). Other sequences give the same inverse.

**Algebraic properties.** $(A^{-1})^{-1} = A$; $(AB)^{-1} = B^{-1}A^{-1}$ (proof: $(AB)(B^{-1}A^{-1}) =
A(BB^{-1})A^{-1} = AA^{-1} = I_n$); $(AB)^{-1} = A^{-1}B^{-1}$ iff $AB = BA$; $(A^\top)^{-1} = (A^{-1})^\top$;
$\det(A^{-1}) = 1/\det A$.

> 🎓 **Exam tip.** When you invert by hand, write the operation next to every arrow ($R_1 \leftarrow R_1 - 3R_2$) and work
> column by column: make the pivot 1, then clear the rest of its column. And remember “shoes and socks”: $(AB)^{-1} =
> B^{-1}A^{-1}$.

---
## 5 · When is a matrix invertible? The invertible matrix theorem and Cramer's rule

> 📍 **Where we are.** We can invert a matrix — when it is invertible. Can we tell *in advance*? Many seemingly different
> properties turn out to be the same property in disguise. This list is one of the most useful summaries of the whole
> linear-algebra part of the course: you will meet each item again in Modules 10–13.

**The invertible matrix theorem.** For a square matrix $A\in\mathbb R^{n\times n}$ the following statements are
**equivalent** (all true or all false):

| # | Statement | What it means | Where in the course |
|---|---|---|---|
| 1 | $A$ is invertible: $\exists A^{-1}$ with $AA^{-1} = A^{-1}A = I$ | the transformation can be undone | this module |
| 2 | $A\mathbf x = \mathbf b$ has a **unique** solution for every $\mathbf b\in\mathbb R^n$ | every system with this matrix is “determined” | Sections 2, 6 |
| 3 | $\operatorname{rref}(A) = I$ | Gauss–Jordan reaches the identity | Section 6 |
| 4 | $\det(A)\neq 0$ | the map does not squash volume to 0 | Module 08 |
| 5 | the columns of $A$ are linearly independent | no column is a combination of the others | Module 11 |
| 6 | the rows of $A$ are linearly independent | no equation is redundant | Module 11 |
| 7 | the columns of $A$ span $\mathbb R^n$ | every $\mathbf b$ can be built from the columns | Module 11 |
| 8 | the rows of $A$ span $\mathbb R^n$ | | Module 11 |
| 9 | $\operatorname{rank}(A) = n$ | “full rank” | Module 11 |
| 10 | 0 is **not** an eigenvalue of $A$ | no direction is sent to $\mathbf 0$ | Module 12 |
| 11 | $A$ is row equivalent to the identity matrix | reachable from $I$ by row operations | Sections 3–4 |
| 12 | $A$ is a product of elementary matrices | | Section 4 |

*In words:* checking any **one** of them tells you all the others. (Statement 1 is listed so the circle is complete:
“$A$ invertible $\iff$ statements 2–12”.) The next cell computes several of them independently, for an invertible
3 × 3 matrix and for a singular one whose third row is the first row minus the second.

In [ ]:
# ── 5.1 Several conditions of the invertible matrix theorem, computed independently ──
def invertibility_checks(M):
    """Evaluate several equivalent conditions of the invertible matrix theorem for a square matrix M."""
    S = sp.Matrix(M)
    n = S.shape[0]
    rng_local = np.random.default_rng(SEED)
    b_random = sp.Matrix(rng_local.integers(-5, 6, n))
    solutions = sp.linsolve((S, b_random), *sp.symbols(f"x1:{n + 1}"))
    unique = len(solutions) == 1 and not list(solutions)[0].free_symbols
    eigenvalues = np.linalg.eigvals(np.array(M, dtype=float))
    return {
        "det(A) ≠ 0": S.det() != 0,
        "rref(A) = I": S.rref()[0] == sp.eye(n),
        "rank(A) = n": S.rank() == n,
        "columns independent (Ax = 0 only for x = 0)": len(S.nullspace()) == 0,
        "rows independent (Aᵀx = 0 only for x = 0)": len(S.T.nullspace()) == 0,
        "Ax = b has one solution (random b)": unique,
        "0 is not an eigenvalue": bool(np.min(np.abs(eigenvalues)) > 1e-10),
    }


A_inv3 = [[3, -2, 1], [1, 2, -1], [2, -1, 3]]
A_sing3 = [[1, 2, 1], [-2, -3, 1], [3, 5, 0]]           # row 3 = row 1 − row 2
table = pd.DataFrame({"invertible A": invertibility_checks(A_inv3), "singular A": invertibility_checks(A_sing3)})
print(table.to_string())
print("\ndet:", sp.Matrix(A_inv3).det(), "and", sp.Matrix(A_sing3).det(),
      "| eigenvalues of the singular matrix:", np.round(np.linalg.eigvals(np.array(A_sing3, float)), 4))

🔍 **What just happened?** For $A = \begin{bmatrix}3&-2&1\\1&2&-1\\2&-1&3\end{bmatrix}$ every condition is `True`
($\det A = 20$); for the singular matrix every one is `False` ($\det = 0$, rank 2, a non-zero vector in the null space,
one eigenvalue equal to 0, and the random system has either no solution or infinitely many). Seven
computations that look unrelated — a determinant, an elimination, a null space, a random right-hand side, eigenvalues —
give the same verdict. That is what “equivalent conditions” means, and it gives you a choice of tests: by hand, the
determinant (small matrices) or elimination (larger ones) are easiest; on a computer, the rank or a condition number
(Section 10).

### 5.2 Cramer's rule

Module 08 gave a formula for the inverse: $A^{-1} = \operatorname{adj}(A)/\det(A)$, where the **adjugate**
$\operatorname{adj}(A)$ is the transposed matrix of cofactors. Writing out $\mathbf x = A^{-1}\mathbf b$ entry by entry
gives **Cramer's rule**: if $A$ is nonsingular,

$$x_i = \frac{\det B_i}{\det A}, \qquad B_i = A \text{ with its } i\text{-th column replaced by } \mathbf b .$$

*In words:* each unknown is a ratio of two determinants: the determinant of $A$ with column $i$ swapped for the
right-hand side, divided by the determinant of $A$. Why: entry $i$ of $\operatorname{adj}(A)\mathbf b$ is

$$x_i = \frac{1}{\det A}\sum_{j=1}^{n} (-1)^{j+i}\,\det\!\big(A_{\setminus j,\setminus i}\big)\,b_j ,$$

*In words:* the sum multiplies each $b_j$ by the cofactor of entry $(j, i)$ — the determinant of $A$ without row $j$ and
column $i$, with the checkerboard sign $(-1)^{j+i}$. That sum is exactly the cofactor expansion of $\det B_i$ down its
column $i$ (which holds $\mathbf b$). The worked example:

$$\begin{aligned} 3x_1 - 2x_2 + x_3 &= -9\\ x_1 + 2x_2 - x_3 &= 5\\ 2x_1 - x_2 + 3x_3 &= -10 \end{aligned}
\qquad \det A = 3\cdot\det\begin{bmatrix}2&-1\\-1&3\end{bmatrix} - (-2)\cdot\det\begin{bmatrix}1&-1\\2&3\end{bmatrix} + 1\cdot\det\begin{bmatrix}1&2\\2&-1\end{bmatrix} = 3\cdot5 + 2\cdot5 + 1\cdot(-5) = 20 .$$

*In words:* expand $\det A$ along the first row (Module 08): the three 2 × 2 minors are 5, 5 and −5, and the signs
alternate $+,-,+$. Replacing column 1 by $\mathbf b = (-9, 5, -10)$ gives
$\det B_1 = -9\cdot5 - (-2)\cdot5 + 1\cdot15 = -20$; similarly $\det B_2 = 40$ and $\det B_3 = -40$, so
$x_1 = -20/20 = -1$, $x_2 = 40/20 = 2$, $x_3 = -40/20 = -2$.

In [ ]:
# ── 5.2 Cramer's rule, exactly, on the 3 × 3 example and on the running 2 × 2 example ──
def cramer_solve(M, rhs):
    """Solve M x = rhs by Cramer's rule x_i = det(B_i) / det(M), with exact SymPy determinants."""
    S, bb = sp.Matrix(M), sp.Matrix(rhs)
    det_S = S.det()
    if det_S == 0:
        raise ValueError("det = 0: Cramer's rule needs a nonsingular matrix")
    solution, dets = [], []
    for i in range(S.shape[1]):
        B_i = S.copy()
        B_i[:, i] = bb                                   # replace column i by the right-hand side
        dets.append(B_i.det())
        solution.append(dets[-1] / det_S)
    return solution, det_S, dets


x_cr, det_A3, dets_B = cramer_solve(A_inv3, [-9, 5, -10])
print("det A =", det_A3, "| det B1, B2, B3 =", dets_B, "| x =", x_cr)
print("np.linalg.solve:", np.linalg.solve(np.array(A_inv3, float), [-9, 5, -10]))
z_cr, det_C2, dets_C = cramer_solve(C_num, b_num)
print("running example: det C =", det_C2, "| det B1, B2 =", dets_C, "| z =", z_cr)

🔍 **What just happened?** Cramer's rule reproduced the worked example: $\det A = 20$, $\det B_1 = -20$, $\det B_2 = 40$,
$\det B_3 = -40$, hence $\mathbf x = (-1, 2, -2)$, the same as `np.linalg.solve` (check the first equation:
$3(-1) - 2(2) + (-2) = -9$ ✓). For the running example, $\det B_1 = \det\begin{bmatrix}-1&2\\3&-5\end{bmatrix} = 5 - 6 =
-1$ and $\det B_2 = \det\begin{bmatrix}3&-1\\1&3\end{bmatrix} = 9 + 1 = 10$, so $\mathbf z = (-1/-17,\ 10/-17) =
(1/17, -10/17)$ once more. Cramer's rule is elegant for 2 × 2 and 3 × 3 systems and useful in proofs (it shows that the
solution changes smoothly with the entries of $A$). For anything bigger it is a bad algorithm — as the next cell shows.

In [ ]:
# ── 5.3 Why nobody solves large systems with Cramer's rule: cofactor determinants cost about n! operations ──
def det_cofactor(M):
    """Determinant by recursive cofactor expansion along the first row (Module 08) — about n! multiplications."""
    n = M.shape[0]
    if n == 1:
        return M[0, 0]
    return sum((-1) ** j * M[0, j] * det_cofactor(np.delete(M[1:], j, axis=1)) for j in range(n))


def cramer_cofactor(M, rhs):
    """Cramer's rule with cofactor determinants: n + 1 determinants of size n."""
    det_M = det_cofactor(M)
    out = []
    for i in range(M.shape[1]):
        B_i = M.copy(); B_i[:, i] = rhs
        out.append(det_cofactor(B_i) / det_M)
    return np.array(out)


rng_cr = np.random.default_rng(SEED + 5)
rows = []
for n in range(3, 8):
    M_rand, b_rand = rng_cr.normal(size=(n, n)), rng_cr.normal(size=n)
    t0 = time.perf_counter(); x_c = cramer_cofactor(M_rand, b_rand); t_cramer = time.perf_counter() - t0
    t0 = time.perf_counter(); x_s = np.linalg.solve(M_rand, b_rand); t_solve = time.perf_counter() - t0
    rows.append(dict(n=n, cramer_ms=1000 * t_cramer, solve_ms=1000 * t_solve, same_answer=np.allclose(x_c, x_s)))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"\n20! = {math.factorial(20):.2e} multiplications for ONE 20 × 20 cofactor determinant;"
      f"  elimination needs about 20³/3 ≈ {20 ** 3 / 3:.0f}")

🔍 **What just happened?** Both methods agree (`same_answer` is `True`), but the time for Cramer's rule with cofactor
determinants grows by roughly a factor $n$ with every extra unknown — from a fraction of a millisecond at $n = 3$ to a few
tenths of a second at $n = 7$, the factorial growth of Module 05 — while `np.linalg.solve` (which uses elimination) stays
around a few hundredths of a millisecond. Exact timings depend on your computer; the trend does not. At $n = 20$ a single
cofactor determinant would need on the order of $20! \approx 2.4\times10^{18}$ multiplications (counting every product
exactly, as Module 08 did, gives about $4\times10^{18}$); elimination needs a few thousand. Even with determinants computed by elimination, Cramer's rule needs $n + 1$ of them, about $n^4/3$
operations against $n^3/3$ for one elimination.

✏️ **Your turn.** Is $\begin{bmatrix}1&2\\2&4\end{bmatrix}$ invertible? Give three different reasons taken from the
table of the invertible matrix theorem, then confirm with `invertibility_checks`.

### 📘 Lecture notes — the invertible matrix theorem and Cramer's rule

**Invertible matrix theorem.** For $A\in\mathbb R^{n\times n}$ the following are equivalent: (1) $A$ invertible;
(2) $A\mathbf x = \mathbf b$ has a unique solution for every $\mathbf b$; (3) $\operatorname{rref}(A) = I$;
(4) $\det A\ne 0$; (5) columns linearly independent; (6) rows linearly independent; (7) columns span $\mathbb R^n$;
(8) rows span $\mathbb R^n$; (9) $\operatorname{rank}A = n$; (10) 0 is not an eigenvalue; (11) $A$ row equivalent to $I$;
(12) $A$ is a product of elementary matrices.

**Algebraic properties.** $(A^{-1})^{-1} = A$, $(AB)^{-1} = B^{-1}A^{-1}$, $(A^\top)^{-1} = (A^{-1})^\top$,
$\det(A^{-1}) = 1/\det A$.

**Cramer's rule.** If $A$ is nonsingular, $x_i = \det B_i/\det A$ with $B_i$ = $A$ with column $i$ replaced by
$\mathbf b$; equivalently $x_i = \frac{1}{\det A}\sum_j(-1)^{j+i}\det(A_{\setminus j,\setminus i})\,b_j$ (from
$A^{-1} = \operatorname{adj}(A)/\det A$).

**Worked example.** $3x_1 - 2x_2 + x_3 = -9$, $x_1 + 2x_2 - x_3 = 5$, $2x_1 - x_2 + 3x_3 = -10$: $\det A = 3\cdot5 -
(-2)\cdot5 + 1\cdot(-5) = 20$, $\det B_1 = -20$, $\det B_2 = 40$, $\det B_3 = -40$ → $\mathbf x = (-1, 2, -2)$.

**Cost.** $n+1$ determinants; with cofactor expansion each costs $\sim n!$ — hopeless beyond $n\approx 10$. Use
elimination ($\sim n^3/3$).

> 🎓 **Exam tip.** If a question asks “is $A$ invertible?”, any one condition settles it — pick the cheapest: for 2 × 2,
> $ad - bc$; if you spot a column that is a multiple or combination of others, you are done (singular). Cramer's rule is
> the quickest way to get **one** unknown of a 2 × 2 or 3 × 3 system.

---
## 6 · Solving without an inverse: Gauss–Jordan elimination on $[A\mid\mathbf b]$

> 📍 **Where we are.** Inverting works only for square, nonsingular matrices. What if $C$ is singular — or not even
> square (over- or under-determined)? The same row reduction still works; we just carry the right-hand side along
> instead of the identity. This procedure is called **Gauss–Jordan elimination**.

Set up the **augmented matrix**

$$[\,A\mid\mathbf b\,] = \left[\begin{array}{cccc|c} A_{11} & A_{12} & \cdots & A_{1m} & b_1\\ A_{21} & A_{22} & \cdots & A_{2m} & b_2\\ \vdots & \vdots & \ddots & \vdots & \vdots\\ A_{n1} & A_{n2} & \cdots & A_{nm} & b_n\end{array}\right].$$

*In words:* one row per equation — the coefficients on the left of the bar, the right-hand side on the right. This single
matrix contains the whole problem, and every elementary operation turns it into a **new system with the same
solutions** (Section 3). If we can make the left-hand side the identity, every equation reads “one unknown = a number”.

The running example, with the operations in the order of the classic paper solution: $\beta = -3$ from row 2 to row 1,
$\alpha = 1/17$ on row 1, $\beta = 5$ from row 1 to row 2, then swap.

In [ ]:
# ── 6.1 Gauss–Jordan on [C | b] for 3z1 + 2z2 = −1, z1 − 5z2 = 3 ──
C_b = np.column_stack([C_num, b_num])
print("[C | b] =\n" + format_matrix(C_b, 2))
R_cb, _ = apply_row_ops(C_b, [("III", 1, 2, -3), ("II", 1, "1/17"), ("III", 2, 1, 5), ("I", 1, 2)], n_coef=2)
print("→ z1 =", R_cb[0, 2], ", z2 =", R_cb[1, 2])
R_tool, piv_tool = gauss_jordan(C_b, n_coef=2)
print("toolkit gauss_jordan gives the same RREF?", np.array_equal(R_tool, R_cb), "| SymPy rref:", sp.Matrix(C_b).rref()[0].tolist())

🔍 **What just happened?** The augmented matrix went $\left[\begin{array}{cc|c}3&2&-1\\1&-5&3\end{array}\right] \to
\left[\begin{array}{cc|c}0&17&-10\\1&-5&3\end{array}\right] \to \left[\begin{array}{cc|c}0&1&-\frac{10}{17}\\1&-5&3\end{array}\right]
\to \left[\begin{array}{cc|c}0&1&-\frac{10}{17}\\1&0&\frac{1}{17}\end{array}\right] \to
\left[\begin{array}{cc|c}1&0&\frac{1}{17}\\0&1&-\frac{10}{17}\end{array}\right]$. Turned back into equations:
$z_1 = 1/17$, $z_2 = -10/17$ — no inverse was ever formed. The order of operations differs from Section 4 (here the
swap comes last) and it does not matter. The toolkit and SymPy's `rref()` reach the same final matrix.

### 6.2 The exercise: a system with no solution

Determine the solution of $-x_1 + 3x_2 = 5$, $x_1 - 3x_2 = 1$ (the parallel lines of Figure 1.2).

In [ ]:
# ── 6.2 Row reduction exposes an impossible equation 0 = 6 ──
P_b = np.array([[-1, 3, 5], [1, -3, 1]])
R_p, _ = apply_row_ops(P_b, [("III", 2, 1, 1)], n_coef=2)        # R2 ← R2 + R1
print("row 2 now says: 0·x1 + 0·x2 =", R_p[1, 2], "→ impossible")
R_pg, piv_p = gauss_jordan(P_b, n_coef=2)
print("toolkit RREF:\n" + format_matrix(R_pg, 2), "\npivot columns:", piv_p)

🔍 **What just happened?** One Type III operation ($R_2 \leftarrow R_2 + R_1$) produced the row $[0\ 0 \mid 6]$, i.e. the
equation $0x_1 + 0x_2 = 6$, which no $(x_1, x_2)$ can satisfy. Since row operations never change the solution set, the
original system has **no solution**: it is **inconsistent** — the parallel lines of Figure 1.2. The toolkit, told to
choose pivots only among the two coefficient columns, stops at $\left[\begin{array}{cc|c}1&-3&-5\\0&0&6\end{array}\right]$
with one pivot (column 1). If we also let the last column supply a pivot — scale row 2 by $\tfrac16$, then clear the $-5$
above it — we reach the true RREF $\left[\begin{array}{cc|c}1&-3&0\\0&0&1\end{array}\right]$, whose second pivot sits in
the **right-hand-side column**: the universal signature of an inconsistent system (Section 6.4 reads it off exactly this way).
(The coefficient matrix has $\det = 0$; Section 2.3.)

### 6.3 Row-reduced echelon form

Gauss–Jordan elimination aims at a standard target shape. A matrix is in **row-reduced echelon form (RREF)** if

1. the first (left-most) non-zero entry in each row is a 1 — the **leading entry** (or **pivot**);
2. in any column with a leading entry, that 1 is the only non-zero value;
3. the leading entry of row $i$ is to the left of the leading entry (if it exists) of row $i + 1$;
4. rows containing only zeros are at the end (bottom) of the matrix.

Every matrix can be brought to RREF by row operations, and the RREF is **unique** (whatever route you take). In each
worked example of this module the final matrix is in RREF, and the ones before it violate one of the rules. Columns with
a pivot are **pivot columns**; the other coefficient columns belong to **free variables** (Section 7). Here are two
5 × 6 matrices in RREF — the first with a pivot in every row, the second with pivots in columns 1, 2 and 4 only:

$$\begin{bmatrix}1&0&0&0&0&2\\0&1&0&0&0&-1\\0&0&1&0&0&3\\0&0&0&1&0&0\\0&0&0&0&1&4\end{bmatrix}\qquad
\begin{bmatrix}1&0&2&0&-1&3\\0&1&4&0&2&-2\\0&0&0&1&5&7\\0&0&0&0&0&0\\0&0&0&0&0&0\end{bmatrix}$$

*In words:* in both, each non-zero row starts with a 1, every pivot column is otherwise zero, pivots step to the right
going down, and zero rows sit at the bottom. The next cell turns the four rules into a checker and tests it on these
matrices, on a third RREF with only two pivots, on four matrices that each break one rule, and on an intermediate matrix
from Section 6.1.

In [ ]:
# ── 6.3 A checker for the four RREF rules ──
def rref_violation(M, tol=1e-12):
    """Return None if M is in row-reduced echelon form, otherwise a string naming the first rule it breaks."""
    M = np.asarray(M, dtype=float)
    last_pivot, seen_zero_row = -1, False
    for i, row in enumerate(M):
        nonzero = np.flatnonzero(np.abs(row) > tol)
        if nonzero.size == 0:
            seen_zero_row = True
            continue
        if seen_zero_row:
            return f"rule 4: a zero row lies above non-zero row {i + 1}"
        p = nonzero[0]
        if abs(row[p] - 1) > tol:
            return f"rule 1: the leading entry of row {i + 1} is {row[p]:g}, not 1"
        if np.sum(np.abs(M[:, p]) > tol) > 1:
            return f"rule 2: pivot column {p + 1} has other non-zero entries"
        if p <= last_pivot:
            return f"rule 3: the pivot of row {i + 1} is not to the right of the pivot above it"
        last_pivot = p
    return None


examples = {
    "5×6, pivot in every row": [[1, 0, 0, 0, 0, 2], [0, 1, 0, 0, 0, -1], [0, 0, 1, 0, 0, 3], [0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 1, 4]],
    "5×6, pivots in cols 1, 2, 4": [[1, 0, 2, 0, -1, 3], [0, 1, 4, 0, 2, -2], [0, 0, 0, 1, 5, 7], [0] * 6, [0] * 6],
    "5×6, pivots in cols 1, 2": [[1, 0, 3, 0, -2, 5], [0, 1, 4, 0, 1, -1], [0] * 6, [0] * 6, [0] * 6],
    "leading entry 2": [[2, 0, 1], [0, 1, 3]],
    "pivot column not cleared": [[1, 3, 0], [0, 1, 2]],
    "pivots step left": [[0, 1, 2], [1, 0, 5]],
    "zero row on top": [[0, 0, 0], [1, 0, 4]],
    "an intermediate step of 6.1": [[0, 17, -10], [1, -5, 3]],
}
for name, M in examples.items():
    verdict = rref_violation(M)
    print(f"{name:30s} → {'RREF ✓' if verdict is None else 'not RREF: ' + verdict}")

🔍 **What just happened?** The two matrices from the definition box and the third one (with only two pivots) pass all
four rules. Each of the four counter-examples fails exactly the rule it was built to break, and the intermediate matrix
$\left[\begin{array}{cc|c}0&17&-10\\1&-5&3\end{array}\right]$ of Section 6.1 fails rule 1 (its first leading entry is
17). Notice that the RREF rules say nothing about the **last** column of an augmented matrix being special — the checker
treats it as an ordinary column, which is why a pivot can land there (Section 6.2).

### 6.4 Reading the answer off the RREF: one, none or infinitely many solutions

Once $[A\mid\mathbf b]$ is in RREF, look at the pivots:

* a pivot in the **right-hand-side column** (a row $[0\ \cdots\ 0\mid c]$ with $c\ne0$) → **no solution**;
* otherwise, a pivot in **every** coefficient column → **exactly one** solution (read it off);
* otherwise (some coefficient column without a pivot) → **infinitely many**: each pivot-less column is a **free
  variable** that can take any value.

This is why only three outcomes exist for any linear system. The next cell classifies a few systems.

**Echelon form vs reduced echelon form.** If you only create zeros *below* each pivot (and do not insist on pivots equal to
1 or on zeros above them), you reach a **row-echelon form** — the target of **Gaussian elimination**. Rules 3 and 4 still
hold, so a square matrix in row-echelon form is **upper triangular**. Gaussian elimination followed by back-substitution
is what computers do (Section 10); Gauss–Jordan elimination continues to the RREF, which is easier to read by eye. The
number of pivots is the same in both forms — Module 11 calls it the **rank**.

In [ ]:
# ── 6.4 Classify a system from the pivots of its reduced augmented matrix ──
def classify_system(A_coef, rhs):
    """'unique', 'none' or 'infinitely many' solutions of A x = rhs, from the pivots of rref([A | rhs])."""
    A_coef = np.asarray(A_coef)
    aug = np.column_stack([A_coef, rhs])
    exact = np.issubdtype(aug.dtype, np.integer)
    R, piv = gauss_jordan(aug.astype(int) if exact else aug.astype(float))      # n_coef left at None: pivots may land in rhs
    m = A_coef.shape[1]
    if m in piv:
        return "none", piv
    return ("unique" if len(piv) == m else "infinitely many"), piv


systems_to_classify = {
    "running example 3z1+2z2=−1, z1−5z2=3": ([[3, 2], [1, -5]], [-1, 3]),
    "parallel lines −x1+3x2=5, x1−3x2=1": ([[-1, 3], [1, -3]], [5, 1]),
    "one equation x + y = 2": ([[1, 1]], [2]),
    "3 equations in 5 unknowns (Section 7)": ([[2, -1, 4, 1, -1], [3, 2, -2, -5, 2], [1, 1, 6, 2, 1]], [4, 9, 7]),
    "Cramer example (3 × 3)": (A_inv3, [-9, 5, -10]),
}
for name, (A_coef, rhs) in systems_to_classify.items():
    kind, piv = classify_system(A_coef, rhs)
    print(f"{name:42s} → {kind:16s} pivot columns {[p + 1 for p in piv]}")

🔍 **What just happened?** The running example and the Cramer example have a pivot in every coefficient column → one
solution each. The parallel-lines system has its second pivot in column 3, the right-hand side → none. The single
equation $x + y = 2$ and the 3 × 5 system have fewer pivots than unknowns (1 of 2, and 3 of 5) → infinitely many
solutions, with 1 and 2 free variables respectively. Section 7 describes those infinite solution sets exactly.

### 🌉 Bridge — How much of each cell type is in this sample? (deconvolution)

Return to the biological question. Let $f_c$ be the fraction of cells of type $c$ and $\lambda_{gc}$ the mean
expression of gene $g$ in a cell of type $c$ (the **signature**). A cell drawn at random from the tissue is of type $c$
with probability $f_c$, so the average expression of gene $g$ over all cells is

$$b_g = \sum_{c} \lambda_{gc}\, f_c \qquad\Longleftrightarrow\qquad \mathbf b = S\,\mathbf f,\quad S_{gc} = \lambda_{gc}.$$

*In words:* the bulk level of each marker is the fraction-weighted average of its level in the pure cell types; with one
marker per cell type, $S$ is a 4 × 4 matrix and $\mathbf b = S\mathbf f$ is a square linear system for the four unknown
fractions. Real deconvolution methods (CIBERSORT, Newman *et al.* 2015) use hundreds of genes and add the constraints
$f_c\ge 0$, $\sum_c f_c = 1$, but they solve this equation. Here the signature is the simulator's true one and the
bulk sample is the SIMULATED pseudo-bulk of Section 0.

In [ ]:
# ── 6.5 Solve the 4 × 4 deconvolution system S f = b (SIMULATED pseudo-bulk, true signature) ──
idx4 = [genes.index(g) for g in markers4]
S4 = rates[markers4].T.to_numpy()                   # rows = marker genes, columns = cell types
b4 = bulk[idx4]
print("signature matrix S (molecules per cell):")
print(pd.DataFrame(S4, index=markers4, columns=cell_types).round(3).to_string())
print("det S =", round(np.linalg.det(S4), 1), "→ invertible; classification:", classify_system(S4, b4)[0])
f_hat = np.linalg.solve(S4, b4)
comparison = pd.DataFrame({"true fraction": f_true.to_numpy(), "estimated": f_hat}, index=cell_types)
print("\n" + comparison.round(4).to_string())
print("sum of estimated fractions:", round(f_hat.sum(), 4))

🔍 **What just happened?** The signature matrix is dominated by its diagonal (each marker is high, about 6–13 molecules
per cell, only in its own cell type, and 0.1–0.5 elsewhere), its determinant is large and the system has a unique
solution. Solving it recovered the fractions well: about 0.364 / 0.258 / 0.196 / 0.160 against the truth
0.364 / 0.273 / 0.205 / 0.159. The small errors (largest for B cells, about 0.015) come from the Poisson noise of 440
simulated cells: the pseudo-bulk levels are averages of random counts, not the exact expected values. The estimated
fractions sum to 0.978 rather than exactly 1 — nothing in the equations forces them to, which is one reason real methods
add that constraint. See [BiologyToML 03 · linear algebra for data](../../../../ml/tutorial/BiologyToML/notebooks/03_linear_algebra_for_data.ipynb) for the same
calculation on blood cell types.

✏️ **Your turn.** Replace the macrophage marker CD68 by LYZ (also a macrophage marker, `CELL_TYPE_MARKERS["macrophage"]`)
and solve again. Do the estimates change? Then use CD3E as the marker for **both** T cells and B cells (two identical
equations): what do `np.linalg.det` and `classify_system` say now, and why?

### 📘 Lecture notes — Gauss–Jordan elimination and row-reduced echelon form

**Gauss–Jordan elimination.** Form the augmented matrix $[A\mid\mathbf b]$; apply elementary row operations (each gives
a system with the same solutions) until the left part is as close to $I$ as possible (the RREF); turn the rows back into
equations. Works for any shape of $A$, invertible or not.

**Worked example.** $\left[\begin{array}{cc|c}3&2&-1\\1&-5&3\end{array}\right]$: Type III with $\beta=-3$ (row 2 to
row 1) → $\left[\begin{array}{cc|c}0&17&-10\\1&-5&3\end{array}\right]$; Type II $\alpha = 1/17$ on row 1; Type III
$\beta = 5$ from row 1 to row 2 → $\left[\begin{array}{cc|c}0&1&-\frac{10}{17}\\1&0&\frac1{17}\end{array}\right]$; swap →
$z_1 = \frac1{17}$, $z_2 = -\frac{10}{17}$.

**Exercise.** $-x_1 + 3x_2 = 5$, $x_1 - 3x_2 = 1$: $R_2 + R_1$ gives $0 = 6$ → **no solution** (parallel lines).

**RREF rules.** (1) leading entry of each non-zero row is 1; (2) a pivot is the only non-zero entry of its column;
(3) pivots move strictly right going down; (4) zero rows at the bottom. The RREF of a matrix is unique.

**Reading solutions.** Pivot in the $\mathbf b$ column → none; pivots in all $m$ coefficient columns → unique;
otherwise infinitely many (free variables = pivot-less columns).

> 🎓 **Exam tip.** Keep fractions exact and write each operation; an exam reduction with decimals almost always goes
> wrong. Before reading off a solution, check the bottom rows: a row $[0 \cdots 0 \mid c]$ with $c\neq 0$ means “no
> solution”, whatever the other rows say.

---
## 7 · Under-determined systems and the parametric vector form

> 📍 **Where we are.** Section 6 showed that a system with free variables has infinitely many solutions. Here we describe
> those infinite sets exactly — as a point plus all combinations of a few directions — and see their geometry: lines,
> planes and their higher-dimensional cousins.

**Under-determined system.** A system with **fewer equations than unknowns**: there is not enough information
(constraints) to pin down a single solution. Instead of a point, the solution set (if it is not empty) is an infinite set
of vectors. With one equation, $x + y = 2$, there are infinitely many pairs: $(1, 1)$, $(0, 2)$, $(2, 0)$, $(-3, 5)$, …

**Geometric interpretation.** Each linear equation is a flat sheet: a line in 2-D, a plane in 3-D, a **hyperplane** in
higher dimensions. The solutions are where all the sheets overlap.

* In 2-D, one equation = one line: all points on the line are solutions.
* Two equations in two unknowns (excluding parallel lines) meet in a single point — e.g. $x - 3y = -3$ and $2x + y = 8$.
* In 3-D, one equation = one plane: all points on the plane are solutions.
* Two planes in 3-D (excluding parallel planes) intersect in a **line** — e.g. $x + y + z = 1$ and $x - z = 0$.

In an under-determined system the intersection is typically a line, a plane, or a higher-dimensional flat set (an
**affine subspace**; Module 10).

In [ ]:
# ── 7.1 One line; two lines meeting in a point; two planes meeting in a line ──
fig = plt.figure(figsize=(11.5, 3.9))
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(*line_points(1, 1, 2), color="tab:blue", lw=2, label="x + y = 2")
pts = np.array([[1, 1], [0, 2], [2, 0], [-1, 3]])
ax1.plot(*pts.T, "ko", ms=5); [ax1.annotate(f"({p[0]}, {p[1]})", p, xytext=(p[0] + 0.15, p[1] + 0.2), fontsize=8) for p in pts]
ax1.set_title("one equation, two unknowns:\na whole line of solutions")
ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(*line_points(1, -3, -3, (-1, 5)), color="tab:purple", lw=2, label="x − 3y = −3")
ax2.plot(*line_points(2, 1, 8, (-1, 5)), color="tab:green", lw=2, label="2x + y = 8")
xy = np.linalg.solve([[1, -3], [2, 1]], [-3, 8])
ax2.plot(*xy, "ko", ms=6); ax2.annotate(f"({xy[0]:.0f}, {xy[1]:.0f})", xy, xytext=(xy[0] + 0.3, xy[1] + 0.4), fontsize=8)
ax2.set_title("two equations, two unknowns:\none point")
for ax, lim in [(ax1, (-2, 4)), (ax2, (-1, 5))]:
    ax.axhline(0, color="0.8", lw=0.8); ax.axvline(0, color="0.8", lw=0.8)
    ax.set_xlim(*lim); ax.set_ylim(*lim); ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(fontsize=7, loc="lower left")

ax3 = fig.add_subplot(1, 3, 3, projection="3d")
g = np.linspace(-1.5, 1.5, 25)
X3, Y3 = np.meshgrid(g, g)
Z_purple = 1 - X3 - Y3                                       # x + y + z = 1  ⇔  z = 1 − x − y
Z_purple[(Z_purple < -1.5) | (Z_purple > 1.5)] = np.nan     # keep the plane inside the box
ax3.plot_surface(X3, Y3, Z_purple, alpha=0.35, color="tab:purple")
ax3.plot_surface(X3, Y3, X3.copy(), alpha=0.35, color="tab:green")          # x − z = 0  ⇔  z = x
t = np.linspace(-0.25, 1.25, 50)                           # keeps the line inside the box
ax3.plot(t, 1 - 2 * t, t, color="tab:red", lw=4, label="line (t, 1 − 2t, t)", zorder=10)
ax3.set_xlabel("x"); ax3.set_ylabel("y"); ax3.set_zlabel("z"); ax3.set_zlim(-1.5, 1.5); ax3.set_ylim(-1.5, 1.5)
ax3.set_title("x + y + z = 1 and x − z = 0:\ntwo planes meet in a line", fontsize=9); ax3.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

for t_val in [0, 0.5, 1]:                                   # every point of the red line solves both equations
    x_, y_, z_ = t_val, 1 - 2 * t_val, t_val
    print(f"t = {t_val}: (x, y, z) = ({x_}, {y_}, {z_}) → x+y+z = {x_ + y_ + z_}, x−z = {x_ - z_}")

🔍 **What just happened?** Left: all four marked points — $(1,1)$, $(0,2)$, $(2,0)$, $(-1,3)$ — lie on $x + y = 2$; one
equation leaves one direction of freedom. Middle: $x - 3y = -3$ and $2x + y = 8$ meet only at $(3, 2)$ (check:
$3 - 6 = -3$ and $6 + 2 = 8$). Right: the purple plane $x + y + z = 1$ and the green plane $z = x$ cut each other along
the red line $(t,\ 1 - 2t,\ t)$; the printout confirms three of its points satisfy both equations. Two equations in three
unknowns leave **one** free parameter $t$.

### 7.2 A bigger example: three equations, five unknowns

$$\begin{aligned}
2x_1 - x_2 + 4x_3 + x_4 - x_5 &= 4\\
3x_1 + 2x_2 - 2x_3 - 5x_4 + 2x_5 &= 9\\
x_1 + x_2 + 6x_3 + 2x_4 + x_5 &= 7
\end{aligned}$$

*In words:* three hyperplanes in $\mathbb R^5$. With fewer equations (three) than unknowns (five) the solution cannot be
a single point — but can we describe the intersection exactly? We row-reduce $[A\mid\mathbf b]$ with the classic sequence
of hand operations: swap rows 1 and 3 (to put a leading 1 on top); clear column 1 ($\beta = -3$ and $\beta = -2$); scale
row 2 by $\alpha = -1$; clear column 2 ($\beta = -1$ into row 1, $\beta = 3$ into row 3); scale row 3 by $\alpha = 1/52$;
clear column 3 ($\beta = 14$ into row 1, $\beta = -20$ into row 2).

In [ ]:
# ── 7.2 Reduce the 3 × 5 system step by step (exact fractions) ──
A35 = np.array([[2, -1, 4, 1, -1], [3, 2, -2, -5, 2], [1, 1, 6, 2, 1]])
b35 = np.array([4, 9, 7])
aug35 = np.column_stack([A35, b35])
print("[A | b] =\n" + format_matrix(aug35, 5))
steps35 = [("I", 1, 3),
           ("III", 2, 1, -3), ("III", 3, 1, -2),
           ("II", 2, -1),
           ("III", 1, 2, -1), ("III", 3, 2, 3),
           ("II", 3, "1/52"),
           ("III", 1, 3, 14), ("III", 2, 3, -20)]
R35, _ = apply_row_ops(aug35, steps35, n_coef=5)
print("\nin RREF?", rref_violation(R35[:, :5]) is None, "| same as the toolkit's RREF?",
      np.array_equal(R35, gauss_jordan(aug35, n_coef=5)[0]))
print("as a fraction, −28/52 =", Fraction(-28, 52), "and −48/52 =", Fraction(-48, 52))

🔍 **What just happened?** The printout follows the hand calculation line by line: after the swap and the first two
Type III steps the matrix is $\left[\begin{array}{ccccc|c}1&1&6&2&1&7\\0&-1&-20&-11&-1&-12\\0&-3&-8&-3&-3&-10\end{array}\right]$;
after clearing column 2 the third row is $[0\ 0\ 52\ 30\ 0 \mid 26]$; dividing by 52 and clearing column 3 gives the RREF

$$\left[\begin{array}{ccccc|c} 1 & 0 & 0 & -\tfrac{12}{13} & 0 & 2\\ 0 & 1 & 0 & -\tfrac{7}{13} & 1 & 2\\ 0 & 0 & 1 & \tfrac{15}{26} & 0 & \tfrac12 \end{array}\right].$$

*In words:* three pivots (columns 1–3) and two pivot-less columns (4 and 5) — two free variables. `Fraction` reduces
every fraction automatically: on paper you may write $-\tfrac{48}{52}$, $\tfrac{30}{52}$ and $\tfrac{26}{52}$, which are
the same numbers as $-\tfrac{12}{13}$, $\tfrac{15}{26}$ and $\tfrac12$.

⚠️ **Common confusion — the $x_4$ coefficient in row 2.** It is $11 - 20\cdot\tfrac{30}{52} = \tfrac{572 - 600}{52} =
-\tfrac{28}{52} = -\tfrac{7}{13}$. It is easy to slip here (writing $-\tfrac{28}{56}$, or simplifying to
$-\tfrac{14}{13}$); the check in Section 7.3 — plugging the solution back into all three equations — catches any such
slip.

**Back to equations.** The RREF says

$$x_1 - \tfrac{12}{13}x_4 = 2,\qquad x_2 - \tfrac{7}{13}x_4 + x_5 = 2,\qquad x_3 + \tfrac{15}{26}x_4 = \tfrac12 .$$

*In words:* the pivot variables $x_1, x_2, x_3$ are expressed through the free variables $x_4, x_5$, which can be
anything. Let $x_4 = s$ and $x_5 = t$ with $s, t\in\mathbb R$; then $x_1 = 2 + \tfrac{12}{13}s$,
$x_2 = 2 + \tfrac{7}{13}s - t$, $x_3 = \tfrac12 - \tfrac{15}{26}s$. Stacking the five unknowns gives the
**parametric vector form**

$$\mathbf x = \begin{bmatrix}x_1\\x_2\\x_3\\x_4\\x_5\end{bmatrix}
= \underbrace{\begin{bmatrix}2\\2\\ \tfrac12\\0\\0\end{bmatrix}}_{\text{a particular solution}}
+\; s\underbrace{\begin{bmatrix}\tfrac{12}{13}\\ \tfrac{7}{13}\\ -\tfrac{15}{26}\\1\\0\end{bmatrix}}_{\text{free direction 1}}
+\; t\underbrace{\begin{bmatrix}0\\-1\\0\\0\\1\end{bmatrix}}_{\text{free direction 2}},\qquad s, t\in\mathbb R .$$

*In words:* every solution is one fixed solution plus any amount $s$ of the first direction plus any amount $t$ of the
second. Every choice of $(s, t)$ gives a solution, and every solution arises this way: the solution set is a
**2-dimensional plane in 5-dimensional space** (not through the origin).

In [ ]:
# ── 7.3 Verify the parametric vector form: particular solution, directions, and 1000 random members ──
p_sol = np.array([Fraction(2), Fraction(2), Fraction(1, 2), Fraction(0), Fraction(0)], dtype=object)
dir1 = np.array([Fraction(12, 13), Fraction(7, 13), Fraction(-15, 26), Fraction(1), Fraction(0)], dtype=object)
dir2 = np.array([Fraction(0), Fraction(-1), Fraction(0), Fraction(0), Fraction(1)], dtype=object)
print("A p    =", [str(v) for v in A35 @ p_sol], " (should be b = [4, 9, 7])")
print("A dir1 =", [str(v) for v in A35 @ dir1], " A dir2 =", [str(v) for v in A35 @ dir2], " (should be 0)")

rng_pv = np.random.default_rng(SEED + 7)
s_vals, t_vals = rng_pv.uniform(-10, 10, 1000), rng_pv.uniform(-10, 10, 1000)
X_members = (p_sol.astype(float)[:, None] + dir1.astype(float)[:, None] * s_vals + dir2.astype(float)[:, None] * t_vals)
residuals = A35 @ X_members - b35[:, None]
print(f"1000 random (s, t): largest |A x − b| = {np.abs(residuals).max():.1e}")

x1_5 = sp.symbols("x1:6")
print("SymPy linsolve:", sp.linsolve((sp.Matrix(A35), sp.Matrix(b35)), *x1_5))

🔍 **What just happened?** Exactly: $A\mathbf p = (4, 9, 7) = \mathbf b$, so $\mathbf p$ is a solution, and both free
directions satisfy $A\mathbf d = \mathbf 0$ — adding any multiple of them does not change $A\mathbf x$. Numerically,
1000 random members of the family all solve the system up to rounding error (about $10^{-14}$). SymPy's `linsolve`
returns the same family written with $x_4$ and $x_5$ as the free symbols: $x_1 = \tfrac{12x_4}{13} + 2$,
$x_2 = \tfrac{7x_4}{13} - x_5 + 2$, $x_3 = \tfrac12 - \tfrac{15x_4}{26}$. Notice the structure — **solution = one
particular solution + vectors that $A$ sends to $\mathbf 0$** — which Section 8 makes precise.

### 7.4 Over-determined systems

The opposite case — **more equations than unknowns** — is common in biology, because we usually measure more things than
we want to estimate. Such a system is typically **inconsistent**: measurement noise makes the equations slightly
incompatible, the RREF shows a pivot in the right-hand-side column, and no exact solution exists.

### 🌉 Bridge — More markers than cell types: from exact solutions to least squares

Use all 12 marker genes of `CELL_TYPE_MARKERS` (three per cell type) for the four unknown fractions: 12 equations, 4
unknowns.

In [ ]:
# ── 7.4 Twelve markers, four cell types: an over-determined, inconsistent system (SIMULATED) ──
markers12 = [g for t in cell_types for g in CELL_TYPE_MARKERS[t]]
S12 = rates[markers12].T.to_numpy()
b12 = bulk[[genes.index(g) for g in markers12]]
kind12, piv12 = classify_system(S12, b12)
print(f"12 × 4 system: {kind12} (pivot columns {[p + 1 for p in piv12]}; column 5 is the right-hand side)")
f_ls, *_ = np.linalg.lstsq(S12, b12, rcond=None)          # the least-squares compromise (Module 15)
residual = S12 @ f_ls - b12
print(pd.DataFrame({"true": f_true.to_numpy(), "4 markers (exact solve)": f_hat, "12 markers (least squares)": f_ls},
                   index=cell_types).round(4).to_string())
print("least-squares residual per marker (molecules per cell):", residual.round(3))

🔍 **What just happened?** With 12 noisy equations and 4 unknowns, Gauss–Jordan elimination finds a pivot in the
right-hand-side column: **no** vector of fractions satisfies all 12 equations exactly. The practical answer is the
**least-squares** solution — the $\mathbf f$ that makes the total squared mismatch $\|S\mathbf f - \mathbf b\|^2$ as
small as possible. Here it gives about 0.360 / 0.277 / 0.201 / 0.161 — closer to the truth overall than the four-marker
solution (largest error 0.004 instead of 0.015), because errors in individual markers partly average out; the remaining
mismatch is at most about 0.12 molecules per cell for any marker. Least squares is derived with calculus in [Module 15](15_multivariable_calculus_dynamics.ipynb)
(normal equations $S^\top S\mathbf f = S^\top\mathbf b$, a square system solved with this module's tools), it is the
heart of linear regression ([Module 27](27_linear_regression_review.ipynb);
[BiologyToML 06 · linear regression and GWAS](../../../../ml/tutorial/BiologyToML/notebooks/06_linear_regression_gwas.ipynb)), and real deconvolution software
solves a constrained version of it.

✏️ **Your turn.** Solve $x_1 + 2x_2 - x_3 = 4$ (one equation, three unknowns) in parametric vector form by hand. How many
free variables are there, and what geometric object is the solution set? Check one member with code.

---
## 8 · Homogeneous systems, the null space and the “−1 trick”

> 📍 **Where we are.** The free directions of Section 7 satisfied $A\mathbf d = \mathbf 0$. Systems whose right-hand side
> is zero are so important that they get their own name; their solution sets are the **null spaces** that reappear in
> Modules 10–13 (and, in this section, in balanced chemical equations).

**Homogeneous system.** $A\mathbf x = \mathbf 0$. It always has at least the **trivial solution** $\mathbf x = \mathbf 0$.
The set of all its solutions is the **null space** (or kernel) of $A$:

$$N(A) = \{\mathbf x\in\mathbb R^m \mid A\mathbf x = \mathbf 0\}.$$

*In words:* the null space collects every input vector that $A$ sends to zero. $A$ is invertible exactly when
$N(A) = \{\mathbf 0\}$ (condition 5 of the invertible matrix theorem).

**All solutions = one solution + the null space.** If $A\mathbf p = \mathbf b$ and $A\mathbf x = \mathbf b$, then

$$A(\mathbf x - \mathbf p) = A\mathbf x - A\mathbf p = \mathbf b - \mathbf b = \mathbf 0
\quad\Longrightarrow\quad \mathbf x = \mathbf p + \mathbf n \text{ with } \mathbf n \in N(A).$$

*In words:* two solutions of the same system differ by a null-space vector, and adding a null-space vector to a solution
gives another solution. That is exactly the parametric vector form of Section 7: the particular solution $\mathbf p$
plus combinations of $\text{dir}_1, \text{dir}_2\in N(A)$. To describe $N(A)$ we need a **basis** — a few vectors
whose combinations give all of it (Module 11 defines bases precisely).

### 8.1 The −1 trick

A quick way to read a null-space basis off the RREF (it works on $A$ alone — no augmented matrix, since the right-hand
side is zero anyway):

1. Put $A$ into RREF and delete all-zero rows; say $k$ rows (pivots) remain and $A$ has $n$ columns.
2. Extend it to an $n\times n$ square matrix by inserting $n - k$ rows of the form $(0, \dots, 0, -1, 0, \dots, 0)$, each
   with its $-1$ in a pivot-less column, placed so that the diagonal becomes $1, \dots, 1$ (pivots) and $-1$ (free
   columns).
3. The columns that have a $-1$ on the diagonal (the free-variable columns) are null-space basis vectors; negate them to
   get the usual form with a $+1$ in the free position.

**Why it works.** In the RREF, row $i$ says $x_{p_i} + \sum_{j\,\text{free}} R_{ij}x_j = 0$ for pivot column $p_i$. Set one
free variable $x_j = 1$ and the others 0: then $x_{p_i} = -R_{ij}$. The negated column $j$ of the extended matrix holds
precisely $-R_{ij}$ in the pivot positions, $+1$ in position $j$ and 0 in the other free positions — that solution.

In [ ]:
# ── 8.1 The −1 trick, implemented (exact fractions; works for pivots in any columns) ──
def minus_one_trick(M, verbose=True):
    """Null-space basis of M from its RREF: extend to n × n with −1 rows at the free columns, read and negate."""
    R, piv = gauss_jordan(np.asarray(M, dtype=object) if any(isinstance(v, Fraction) for v in np.ravel(M))
                          else np.asarray(M, dtype=int))
    n = R.shape[1]
    nonzero_rows = R[:len(piv)]                                       # step 1: drop the zero rows
    extended = np.empty((n, n), dtype=object)
    for col in range(n):                                              # step 2: row `col` of the extension
        if col in piv:
            extended[col] = nonzero_rows[piv.index(col)]
        else:
            extended[col] = [Fraction(-1) if q == col else Fraction(0) for q in range(n)]
    free = [col for col in range(n) if col not in piv]
    basis = [-extended[:, j] for j in free]                           # step 3: negate the −1 columns
    if verbose:
        print("RREF without zero rows:\n" + format_matrix(nonzero_rows))
        print("extended to n × n (diagonal: 1 = pivot, −1 = free):\n" + format_matrix(extended))
        for j, v in zip(free, basis):
            print(f"column {j + 1} has −1 on the diagonal → negated: {[str(x) for x in v]}")
    return basis


print("Example 1: A = [[1, 2, 3], [0, 1, 4]]")
basis_1 = minus_one_trick([[1, 2, 3], [0, 1, 4]])
print("\nExample 2: A = [[1, 2, 3, 4], [0, 1, 4, 5]]")
basis_2 = minus_one_trick([[1, 2, 3, 4], [0, 1, 4, 5]])

🔍 **What just happened?** **Example 1.** $A = \begin{bmatrix}1&2&3\\0&1&4\end{bmatrix}$ has RREF
$\begin{bmatrix}1&0&-5\\0&1&4\end{bmatrix}$ ($R_1 \leftarrow R_1 - 2R_2$: $3 - 8 = -5$). Here $n = 3$, $k = 2$, so we add one
row $[0, 0, -1]$; column 3 then reads $(-5, 4, -1)$, and negating gives the null-space basis vector $(5, -4, 1)$: the free
variable $x_3 = 1$ forces $x_1 = 5$, $x_2 = -4$. **Example 2.** $A = \begin{bmatrix}1&2&3&4\\0&1&4&5\end{bmatrix}$ has
RREF $\begin{bmatrix}1&0&-5&-6\\0&1&4&5\end{bmatrix}$; adding $[0,0,-1,0]$ and $[0,0,0,-1]$ puts $-1$s in columns 3 and 4,
which read $(-5, 4, -1, 0)$ and $(-6, 5, 0, -1)$; negated, $(5, -4, 1, 0)$ and $(6, -5, 0, 1)$. So

$$N(A) = \operatorname{span}\left\{\begin{bmatrix}5\\-4\\1\\0\end{bmatrix}, \begin{bmatrix}6\\-5\\0\\1\end{bmatrix}\right\}.$$

*In words:* every solution of $A\mathbf x = \mathbf 0$ is $s(5,-4,1,0) + t(6,-5,0,1)$ for some numbers $s, t$ — the
null space is a plane through the origin of $\mathbb R^4$. (A column read *before* negation, such as $(-5, 4, -1)$, is
also in the null space — it is just $-1$ times the basis vector.)

In [ ]:
# ── 8.2 Check: A v = 0 for every basis vector, and SymPy's nullspace() agrees ──
for name, M, basis in [("Example 1", [[1, 2, 3], [0, 1, 4]], basis_1), ("Example 2", [[1, 2, 3, 4], [0, 1, 4, 5]], basis_2)]:
    products = [[str(v) for v in np.array(M) @ vec] for vec in basis]
    sympy_basis = [list(v) for v in sp.Matrix(M).nullspace()]
    print(f"{name}: A·v = {products} | SymPy nullspace: {sympy_basis}")
print("Section 7: the −1 trick on the 3 × 5 matrix gives", [[str(x) for x in v] for v in minus_one_trick(A35, verbose=False)])

🔍 **What just happened?** Every basis vector is sent to zero, and SymPy's `nullspace()` returns exactly the same vectors
($[5, -4, 1]$; $[5, -4, 1, 0]$ and $[6, -5, 0, 1]$) — SymPy uses the same recipe: one basis vector per free variable,
with a 1 in that position. Applied to the 3 × 5 matrix of Section 7 the trick returns $(\tfrac{12}{13}, \tfrac{7}{13},
-\tfrac{15}{26}, 1, 0)$ and $(0, -1, 0, 0, 1)$ — precisely the two free directions of the parametric vector form.

### 🌉 Bridge — Balancing chemical reactions is a null-space problem

Photosynthesis turns carbon dioxide and water into glucose and oxygen: $x_1\,\mathrm{CO_2} + x_2\,\mathrm{H_2O} \to
x_3\,\mathrm{C_6H_{12}O_6} + x_4\,\mathrm{O_2}$. Atoms are neither created nor destroyed, so each element gives one
equation (atoms on the left = atoms on the right):

$$\text{C: } x_1 = 6x_3,\qquad \text{H: } 2x_2 = 12x_3,\qquad \text{O: } 2x_1 + x_2 = 6x_3 + 2x_4 .$$

*In words:* each CO₂ carries one carbon and two oxygens, each H₂O two hydrogens and one oxygen, each glucose 6 C, 12 H
and 6 O, each O₂ two oxygens. Moving everything to the left gives a homogeneous system $A\mathbf x = \mathbf 0$ whose
columns are the molecules (products with a minus sign) and whose rows are the elements. Any null-space vector is a
balanced equation; we want the one with the smallest whole numbers.

In [ ]:
# ── 8.3 Balance photosynthesis and glucose respiration with the −1 trick ──
def balance(element_matrix, names):
    """Smallest whole-number balancing coefficients from the (1-dimensional) null space of the element matrix."""
    (v,) = minus_one_trick(element_matrix, verbose=False)
    scale = math.lcm(*[x.denominator for x in v])
    coeffs = [int(x * scale) for x in v]
    return dict(zip(names, coeffs))


#                 CO2  H2O  C6H12O6  O2       (products enter with a minus sign)
photosynthesis = [[1,   0,   -6,      0],     # carbon
                  [0,   2,  -12,      0],     # hydrogen
                  [2,   1,   -6,     -2]]     # oxygen
print("photosynthesis:", balance(photosynthesis, ["CO2", "H2O", "C6H12O6", "O2"]))

#              C6H12O6  O2  CO2  H2O
respiration = [[6,      0,  -1,   0],          # carbon
               [12,     0,   0,  -2],          # hydrogen
               [6,      2,  -2,  -1]]          # oxygen
coeffs = balance(respiration, ["C6H12O6", "O2", "CO2", "H2O"])
print("respiration:   ", coeffs)
print("check oxygen in respiration:", 6 * coeffs["C6H12O6"] + 2 * coeffs["O2"], "=", 2 * coeffs["CO2"] + coeffs["H2O"])

🔍 **What just happened?** Each element matrix has rank 3 and 4 columns, so its null space is a line through the origin
(one free variable) and the −1 trick returns one basis vector; multiplying by the least common multiple of its
denominators gives whole numbers. Photosynthesis: $6\,\mathrm{CO_2} + 6\,\mathrm{H_2O} \to \mathrm{C_6H_{12}O_6} +
6\,\mathrm{O_2}$. Glucose respiration — the same reaction run backwards inside mitochondria:
$\mathrm{C_6H_{12}O_6} + 6\,\mathrm{O_2} \to 6\,\mathrm{CO_2} + 6\,\mathrm{H_2O}$ (oxygen: $6 + 12 = 18$ atoms on each
side). Any multiple, e.g. $(12, 12, 2, 12)$, is equally balanced — that is the “line” of solutions; chemists simply pick
the smallest whole numbers. For reactions with more molecules than elements + 1, the null space can be 2-dimensional: two
independent balanced reactions hide in one equation.

✏️ **Your turn.** Nitrogen-fixing bacteria turn N₂ into ammonia with the enzyme nitrogenase (whose real reaction also
involves electrons, ATP and a H₂ by-product), and industry does it with the Haber–Bosch process; the simplified atom balance
N₂ + H₂ → NH₃ is a textbook exercise. Write the element matrix (rows N and H, columns N₂, H₂, NH₃) and
balance it with `balance`. (Expected answer: 1, 3, 2.)

### 📘 Lecture notes — under-determined and homogeneous systems, the null space

**Under-determined system.** Fewer equations than unknowns: not enough constraints for a unique solution; the solution
set (if not empty) is infinite — a line, plane or higher-dimensional flat set. Example: $x + y = 2$. Geometry: each
equation is a line (2-D), plane (3-D) or hyperplane; two planes in 3-D meet in a line ($x + y + z = 1$, $x - z = 0$:
the line $(t, 1-2t, t)$).

**Worked example (3 × 5).** $2x_1 - x_2 + 4x_3 + x_4 - x_5 = 4$, $3x_1 + 2x_2 - 2x_3 - 5x_4 + 2x_5 = 9$,
$x_1 + x_2 + 6x_3 + 2x_4 + x_5 = 7$. RREF: $x_1 - \tfrac{12}{13}x_4 = 2$, $x_2 - \tfrac{7}{13}x_4 + x_5 = 2$,
$x_3 + \tfrac{15}{26}x_4 = \tfrac12$. Free $x_4 = s$, $x_5 = t$:
$\mathbf x = (2, 2, \tfrac12, 0, 0) + s(\tfrac{12}{13}, \tfrac{7}{13}, -\tfrac{15}{26}, 1, 0) + t(0, -1, 0, 0, 1)$ — a
2-D plane in $\mathbb R^5$.

**Over-determined system.** More equations than unknowns; usually inconsistent → least squares (Module 15).

**Homogeneous system, null space.** $A\mathbf x = \mathbf 0$ always has $\mathbf x = \mathbf 0$;
$N(A) = \{\mathbf x: A\mathbf x = \mathbf 0\}$. All solutions of $A\mathbf x = \mathbf b$ = one particular solution + $N(A)$.

**The −1 trick.** RREF without zero rows → insert rows $(0,\dots,-1,\dots,0)$ so the diagonal is $1,\dots,1,-1,\dots,-1$ →
the columns with $-1$ on the diagonal, negated, form a null-space basis. Examples:
$\begin{bmatrix}1&2&3\\0&1&4\end{bmatrix}\to(5,-4,1)$;
$\begin{bmatrix}1&2&3&4\\0&1&4&5\end{bmatrix}\to(5,-4,1,0),\ (6,-5,0,1)$.

> 🎓 **Exam tip.** Count first: (number of unknowns) − (number of pivots) = number of free variables = number of null-space
> basis vectors. Then check each basis vector by multiplying with $A$ — the result must be exactly $\mathbf 0$.

---
## 9 · Steady states of a metabolic network

> 📍 **Where we are.** Null spaces are not just an algebra exercise: in systems biology the null space of the
> **stoichiometric matrix** is the set of all possible steady states of a metabolic network.

A metabolic network with metabolites $A, B, C, D$ and reactions $v_1, \dots, v_6$ is summarised by its **stoichiometric
matrix** $S$: rows = metabolites, columns = reactions, entry $S_{ij}$ = how many molecules of metabolite $i$ reaction $j$
produces (positive) or consumes (negative). If $\mathbf v$ holds the reaction rates (**fluxes**), the concentrations
change as

$$\frac{d\mathbf c}{dt} = S\,\mathbf v, \qquad \text{steady state:}\quad S\mathbf v = \mathbf 0 .$$

*In words:* each metabolite's level rises by what the reactions produce and falls by what they consume; at **steady
state** production and consumption balance for every metabolite at once, which is a homogeneous linear system for the
fluxes. Its solution set — the null space of $S$ — is the space of all steady-state flux distributions.

Our **toy** pathway (illustrative, not a real network): metabolite A is taken up ($v_1$), converted either to B ($v_2$) or
to C ($v_3$); both are converted to D ($v_5$: B → D, $v_6$: C → D); D is exported ($v_4$). It has the shape of many real
branch points, where two enzymes or two routes connect the same start and end.

In [ ]:
# ── 9.1 The toy pathway: stoichiometric matrix (metabolites × reactions) and a drawing ──
reactions = ["v1: uptake → A", "v2: A → B", "v3: A → C", "v4: D → export", "v5: B → D", "v6: C → D"]
metabolites = ["A", "B", "C", "D"]
S_net = np.array([[1, -1, -1,  0,  0,  0],      # A: made by v1, used by v2 and v3
                  [0,  1,  0,  0, -1,  0],      # B: made by v2, used by v5
                  [0,  0,  1,  0,  0, -1],      # C: made by v3, used by v6
                  [0,  0,  0, -1,  1,  1]])     # D: made by v5 and v6, used by v4
print(pd.DataFrame(S_net, index=metabolites, columns=[r.split(":")[0] for r in reactions]).to_string())
print("rank S =", np.linalg.matrix_rank(S_net), "| reactions =", S_net.shape[1])

edges = [("in", "A"), ("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "out")]
pos = {"in": (-2, 0), "A": (-1, 0), "B": (0, 0.8), "C": (0, -0.8), "D": (1, 0), "out": (2, 0)}
labels = {("in", "A"): "v1", ("A", "B"): "v2", ("A", "C"): "v3", ("B", "D"): "v5", ("C", "D"): "v6", ("D", "out"): "v4"}
fig, ax = plt.subplots(figsize=(6.4, 2.8))
draw_graph(edges, directed=True, pos=pos, ax=ax, edge_labels=labels,
           node_colors={"in": "#eeeeee", "out": "#eeeeee"}, title="toy branched pathway (illustrative)")
plt.show()

🔍 **What just happened?** Each column of $S$ is one reaction: $v_2$ (A → B) has $-1$ in row A and $+1$ in row B; the
uptake $v_1$ only produces A and the export $v_4$ only consumes D. The matrix has 4 rows, 6 columns and rank 4, so the
homogeneous system $S\mathbf v = \mathbf 0$ has $6 - 4 = 2$ free variables: a 2-dimensional space of steady states. The
drawing shows the branch point at A and the merge at D.

In [ ]:
# ── 9.2 All steady states: the null space of S, and a few flux distributions ──
flux_basis = minus_one_trick(S_net)
print("SymPy nullspace:", [list(v) for v in sp.Matrix(S_net).nullspace()])
route_B, route_C = [np.array(v, dtype=float) for v in flux_basis]
examples_flux = {"route via B": route_B, "route via C": route_C,
                 "uptake 10, 70 % via B": 7 * route_B + 3 * route_C}
for name, v in examples_flux.items():
    print(f"{name:22s} v = {v} → S v = {S_net @ v}")

fig, ax = plt.subplots(figsize=(7.5, 3.2))
width = 0.27
for k, (name, v) in enumerate(examples_flux.items()):
    ax.bar(np.arange(6) + (k - 1) * width, v, width, label=name)
ax.set_xticks(range(6)); ax.set_xticklabels([r.split(":")[0] for r in reactions])
ax.set_xlabel("reaction"); ax.set_ylabel("flux (illustrative units)")
ax.set_title("three steady-state flux distributions of the toy pathway (all satisfy S v = 0)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** Because the branch-specific reactions $v_5$ and $v_6$ are the last two columns, they are the
free variables, and the −1 trick returns two biologically readable basis vectors: $(1, 1, 0, 1, 1, 0)$ — one unit of
flux taken up, sent through B and exported — and $(1, 0, 1, 1, 0, 1)$ — the same through C. Every steady state is a
combination $s\cdot(\text{route via B}) + t\cdot(\text{route via C})$; for example 7 units via B and 3 via C gives
uptake = export = 10, and $S\mathbf v = \mathbf 0$ for all three bar groups. Linear algebra alone cannot say *which*
steady state the cell chooses: that needs extra biology.

### 🌉 Bridge — Flux balance analysis

Genome-scale metabolic models have thousands of reactions and a null space of hundreds of dimensions. **Flux balance
analysis** (FBA; see Orth, Thiele & Palsson 2010, “What is flux balance analysis?”) keeps the steady-state equations
$S\mathbf v = \mathbf 0$, adds bounds such as $0 \le v_j \le v_j^{\max}$ (irreversible reactions, measured uptake
rates), and picks the steady state that maximises a biological objective — typically the flux into biomass, i.e. growth
— by linear programming. Everything starts from the null space you just computed. Module 10 shows that this set of
steady states is a **subspace**, and [Module 11](11_independence_basis_rank.ipynb) explains why its dimension is
“number of reactions − rank of $S$”.

✏️ **Your turn.** Add a reaction $v_7$: B → C to the network (a new column of $S$). What is the rank of the new $S$, how
many basis vectors does the −1 trick return, and can you interpret the new one? (Hint: it may contain a negative flux —
a reaction running backwards — and be a **cycle**.)

---
## 10 · Numerical practice: `solve`, not `inv`

> 📍 **Where we are.** On paper we write $\mathbf x = A^{-1}\mathbf b$. In code there is a better way — and knowing why
> protects you from silent numerical errors in every later module.

`np.linalg.solve(A, b)` performs elimination on $[A\mid\mathbf b]$ (in a factorised form called LU decomposition) and
never builds $A^{-1}$. `np.linalg.inv(A) @ b` first computes the whole inverse — $n$ systems instead of one — and then
multiplies. Both cost on the order of $n^3$ operations, but `solve` does less work and loses fewer digits to rounding.
How many digits can be lost is measured by the **condition number** of $A$, roughly “how much an error in $\mathbf b$ can
be amplified in $\mathbf x$”: for a singular matrix it is infinite, for a nearly singular one huge.

In [ ]:
# ── 10.1 Speed and accuracy of solve vs inv ──
def fastest_time(f, repeats=3):
    """Shortest of a few run times of f() in seconds."""
    best = np.inf
    for _ in range(repeats):
        t0 = time.perf_counter(); f(); best = min(best, time.perf_counter() - t0)
    return best


rng_num = np.random.default_rng(SEED + 10)
for n in [200, 400, 800]:
    M_big = rng_num.normal(size=(n, n)) + n ** 0.5 * np.eye(n)
    b_big = rng_num.normal(size=n)
    print(f"n = {n}: solve {1000 * fastest_time(lambda: np.linalg.solve(M_big, b_big)):6.1f} ms | "
          f"inv @ b {1000 * fastest_time(lambda: np.linalg.inv(M_big) @ b_big):6.1f} ms")

n_h = 12                                                     # the Hilbert matrix H_ij = 1/(i + j − 1): nearly singular
H = 1.0 / (np.arange(1, n_h + 1)[:, None] + np.arange(1, n_h + 1)[None, :] - 1)
x_true = np.ones(n_h)
b_h = H @ x_true
for name, x_est in [("solve", np.linalg.solve(H, b_h)), ("inv @ b", np.linalg.inv(H) @ b_h)]:
    print(f"{name:8s}: residual ‖Hx − b‖ = {np.linalg.norm(H @ x_est - b_h):.1e} | error ‖x − x_true‖ = {np.linalg.norm(x_est - x_true):.1e}")
print(f"condition number of H: {np.linalg.cond(H):.1e}; of the deconvolution matrix S: {np.linalg.cond(S4):.2f}")

🔍 **What just happened?** For random well-behaved matrices `solve` is faster than `inv` followed by a product, and the
gap grows with $n$ (exact milliseconds depend on your computer). On the 12 × 12 **Hilbert matrix**, a classic nearly
singular matrix with condition number about $10^{16}$, `solve` returns a vector whose residual $\|H\mathbf x - \mathbf
b\|$ is at the level of rounding error, while going through the inverse leaves a residual orders of magnitude larger.
**Neither** recovers $\mathbf x_{\text{true}} = (1, \dots, 1)$ exactly: the error $\|\mathbf x - \mathbf x_{\text{true}}\|$
is about 0.45 for `solve` and about 9 for the inverse route — with a condition number near $10^{16}$ even the best
algorithm loses most of the 16 digits a float carries (the precise numbers vary between computers). Our deconvolution matrix, in contrast, has a condition
number of about 2: a well-posed biological measurement. The rule for the rest of the course: **write $A^{-1}\mathbf b$ on
paper, call `np.linalg.solve(A, b)` in code, and look at the condition number when results look strange.**

✏️ **Your turn.** Make the deconvolution problem ill-conditioned: build a 2 × 2 signature for two cell types whose
marker profiles are almost identical, e.g. `[[10, 9.5], [9.6, 10]]`, compute `np.linalg.cond`, and see how much the
estimated fractions move when you change the bulk values by 1 %.

## 📌 Key takeaways

- A system of $n$ linear equations in $m$ unknowns is one matrix equation $C\mathbf x = \mathbf b$ ($C$ is $n\times m$).
  It has **exactly one, no, or infinitely many** solutions — in 2-D: lines that cross, are parallel, or coincide.
- A square matrix $A$ is **invertible** if some $A^{-1}$ satisfies $AA^{-1} = A^{-1}A = I$; then $\mathbf x = A^{-1}\mathbf b$
  is the unique solution. For 2 × 2: $A^{-1} = \frac{1}{ad-bc}\begin{bmatrix}d&-b\\-c&a\end{bmatrix}$; the running
  example has $\det = -17$ and $\mathbf z = (1/17, -10/17)$.
- **Elementary row operations** (swap, scale by $\lambda\ne0$, add a multiple of another row) keep the solution set; each
  is a left multiplication by an **elementary matrix**, and each can be undone.
- **Gauss–Jordan elimination**: $[A\mid I]\to[I\mid A^{-1}]$ inverts a matrix; $[A\mid\mathbf b]\to\operatorname{rref}$
  solves any system. A row $[0\cdots0\mid c\ne0]$ means no solution; pivot-less columns are free variables.
- $(AB)^{-1} = B^{-1}A^{-1}$, $(A^\top)^{-1} = (A^{-1})^\top$, $\det(A^{-1}) = 1/\det A$; invertible ⇔ product of
  elementary matrices ⇔ $\det\ne0$ ⇔ $\operatorname{rref} = I$ ⇔ independent columns ⇔ rank $n$ ⇔ 0 not an eigenvalue
  (the **invertible matrix theorem**).
- **Cramer's rule** $x_i = \det B_i/\det A$ is neat for 2 × 2 and 3 × 3, hopeless for large $n$.
- Solutions of an under-determined system: **particular solution + null space**, written in parametric vector form;
  the 3 × 5 example's solution set is a 2-D plane in $\mathbb R^5$.
- The **null space** $\{\mathbf x : A\mathbf x = \mathbf 0\}$ has a basis readable by the **−1 trick**; it balances
  chemical equations (photosynthesis: 6, 6, 1, 6) and describes all steady states $S\mathbf v = \mathbf 0$ of a metabolic
  network.
- In code use `np.linalg.solve`, never `inv`, and watch the condition number.

---
## ✅ Checkpoint

1. Write $2x_1 - x_2 = 4$, $x_1 + 3x_2 = 9$ as $C\mathbf x = \mathbf b$, invert $C$ with the 2 × 2 formula and solve.
   <details><summary>Answer</summary>$C = \begin{bmatrix}2&-1\\1&3\end{bmatrix}$, $\mathbf b = (4, 9)$. $\det C = 2\cdot3 - (-1)\cdot1 = 7$, $C^{-1} = \frac17\begin{bmatrix}3&1\\-1&2\end{bmatrix}$, $\mathbf x = \frac17(3\cdot4 + 1\cdot9,\ -4 + 2\cdot9) = \frac17(21, 14) = (3, 2)$. Check: $6 - 2 = 4$ ✓, $3 + 6 = 9$ ✓.</details>
2. Solve the same system with Cramer's rule.
   <details><summary>Answer</summary>$\det B_1 = \det\begin{bmatrix}4&-1\\9&3\end{bmatrix} = 12 + 9 = 21$, $\det B_2 = \det\begin{bmatrix}2&4\\1&9\end{bmatrix} = 18 - 4 = 14$; $x_1 = 21/7 = 3$, $x_2 = 14/7 = 2$.</details>
3. Write the 3 × 3 elementary matrix that adds $-4\times$ row 1 to row 3, and its inverse.
   <details><summary>Answer</summary>$\begin{bmatrix}1&0&0\\0&1&0\\-4&0&1\end{bmatrix}$; its inverse adds $+4\times$ row 1 back: $\begin{bmatrix}1&0&0\\0&1&0\\4&0&1\end{bmatrix}$. Their product is $I_3$.</details>
4. Why is “multiply a row by 0” not an elementary operation?
   <details><summary>Answer</summary>It cannot be undone: the equation becomes $0 = 0$ and its information is lost, so the new system can have more solutions than the old one. Elementary operations must keep the solution set, which requires reversibility (scale by $1/\lambda$ needs $\lambda\neq0$).</details>
5. Prove that $(ABC)^{-1} = C^{-1}B^{-1}A^{-1}$ for invertible $A, B, C$.
   <details><summary>Answer</summary>$(ABC)(C^{-1}B^{-1}A^{-1}) = AB(CC^{-1})B^{-1}A^{-1} = A(BB^{-1})A^{-1} = AA^{-1} = I$, and similarly in the other order. (Or: apply $(XY)^{-1} = Y^{-1}X^{-1}$ with $X = AB$, $Y = C$, then once more to $(AB)^{-1}$.)</details>
6. Is $\begin{bmatrix}1&2\\2&4\end{bmatrix}$ invertible? Give three reasons.
   <details><summary>Answer</summary>No. $\det = 4 - 4 = 0$; the second column is twice the first (columns dependent); row reduction gives $\begin{bmatrix}1&2\\0&0\end{bmatrix} \ne I$ (rank 1); and $A\mathbf x = (1, 0)$ has no solution, while $A\mathbf x = (1, 2)$ has infinitely many.</details>
7. Row-reduce $\left[\begin{array}{cc|c}1&2&3\\2&4&7\end{array}\right]$. How many solutions?
   <details><summary>Answer</summary>$R_2 \leftarrow R_2 - 2R_1$ gives $[0\ 0\mid 1]$, i.e. $0 = 1$: no solution (parallel lines $x + 2y = 3$ and $x + 2y = 3.5$).</details>
8. Which of these is in RREF: (a) $\begin{bmatrix}1&0&2\\0&1&3\end{bmatrix}$, (b) $\begin{bmatrix}1&4&0\\0&1&0\end{bmatrix}$, (c) $\begin{bmatrix}0&1&0\\1&0&0\end{bmatrix}$?
   <details><summary>Answer</summary>Only (a). (b) breaks rule 2 (the 4 sits above the pivot of row 2); (c) breaks rule 3 (the pivot of row 2 is to the left of the pivot of row 1).</details>
9. Write the solutions of $x_1 + 2x_2 - x_3 = 4$ in parametric vector form. What geometric object is it?
   <details><summary>Answer</summary>Pivot $x_1$, free $x_2 = s$, $x_3 = t$: $\mathbf x = (4, 0, 0) + s(-2, 1, 0) + t(1, 0, 1)$ — a plane in $\mathbb R^3$ (not through the origin).</details>
10. Use the −1 trick to find the null space of $\begin{bmatrix}1&0&-2\\0&1&3\end{bmatrix}$.
    <details><summary>Answer</summary>Already in RREF; add the row $[0, 0, -1]$; column 3 reads $(-2, 3, -1)$; negated: $(2, -3, 1)$. Check: $2 - 2 = 0$, $-3 + 3 = 0$ ✓. $N(A) = \operatorname{span}\{(2, -3, 1)\}$.</details>
11. Balance $x_1\,\mathrm{N_2} + x_2\,\mathrm{H_2} \to x_3\,\mathrm{NH_3}$.
    <details><summary>Answer</summary>N: $2x_1 = x_3$; H: $2x_2 = 3x_3$. Element matrix $\begin{bmatrix}2&0&-1\\0&2&-3\end{bmatrix}$, null space spanned by $(\tfrac12, \tfrac32, 1)$, i.e. $(1, 3, 2)$: $\mathrm{N_2} + 3\,\mathrm{H_2}\to 2\,\mathrm{NH_3}$.</details>
12. In the toy metabolic network, why is the space of steady states 2-dimensional, and what does a basis vector mean biologically?
    <details><summary>Answer</summary>$S$ has 6 columns (reactions) and 4 pivots (rank 4), so $S\mathbf v = \mathbf 0$ has $6 - 4 = 2$ free variables. The −1 trick basis vectors are the two routes from uptake to export (via B and via C); every steady state is a combination of the two routes.</details>

## 🧠 Questions & discussion

1. In deconvolution, two closely related cell states (say naive and memory T cells) have almost the same marker profile.
   What happens to the determinant and the condition number of the signature matrix, and to the reliability of the
   estimated fractions? What experimental choices could fix it?
2. A genomics study measures 20,000 genes in 50 patients and wants one coefficient per gene. Is that system under- or
   over-determined? What does Section 7 say about its solutions, and why is that a problem for interpretation?
3. The steady-state flux space of a genome-scale metabolic model has hundreds of dimensions. Why does flux balance
   analysis need an *objective* (such as maximal growth)? Is “the cell maximises growth” always a good assumption?
4. Exact fractions (SymPy, `Fraction`) never make rounding errors, so why do scientific programs almost always use
   floating point?
5. Which conditions of the invertible matrix theorem are easiest to check by hand, which by computer, and which give
   the most *insight* into why a particular biological measurement cannot be inverted?

## 🚀 Bonus — going further

- **LU decomposition.** `scipy.linalg.lu(A)` returns $P, L, U$ with $A = PLU$: Gaussian elimination with every Type III
  multiplier stored in $L$ and the row swaps in $P$. Factor the 3 × 3 Cramer matrix and relate $L$ to the elementary
  matrices of Section 4.
- **Partial pivoting.** Solve $\begin{bmatrix}10^{-20}&1\\1&1\end{bmatrix}\mathbf x = (1, 2)$ in floating point with your own
  elimination *without* swapping rows, then with the largest pivot first (what the toolkit's `gauss_jordan` does for float
  input). Why does the first version fail?
- **Flux balance analysis in ten lines.** With `scipy.optimize.linprog`, maximise the export flux $v_4$ of the toy
  pathway subject to $S\mathbf v = \mathbf 0$, $0\le v_j\le 10$ and an uptake bound $v_1\le 6$. Then cap the B branch,
  $v_5\le 2$: where does the rest of the flux go?
- **Watch elimination happen.** Gilbert Strang's MIT 18.06 lectures on elimination, inverses and $A = LU$
  (https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/) teach this module's material with a famous blackboard
  style.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            09 · Linear systems and matrix inverses
Input:             typed-in systems (2 × 2 running example, 3 × 3 Cramer example, 3 × 5 under-determined system);
                   SIMULATED pseudo-bulk of 440 cells (simulate_cell_counts); toy stoichiometric matrix (4 × 6);
                   element matrices of photosynthesis and respiration
Output:            inverses, solutions, RREFs, parametric vector forms, null-space bases, cell-type fractions,
                   balanced equations, steady-state flux bases
Method:            2 × 2 inverse formula; elementary matrices; Gauss–Jordan on [A | I] and [A | b]; Cramer's rule;
                   the −1 trick; np.linalg.solve / lstsq
Key equation:      C x = b  ⇔  x = C⁻¹ b (C invertible);  E_k ⋯ E_1 [A | I] = [I | A⁻¹];  solutions = p + N(A)
Key parameters:    which marker genes (4 vs 12); the order of row operations (does not change the RREF)
What changed when I modified …  (e.g. another marker gene; a near-singular signature; an extra reaction v7)
One failure mode:  a singular or ill-conditioned matrix (det ≈ 0, huge condition number) → no unique / no reliable solution
Connection to 02-680: linear systems and matrix inverses (elementary operations, inversion procedure, invertible
                   matrix theorem, Cramer's rule, Gauss–Jordan, under-determined systems, RREF, −1 trick)
```

## 📚 References

- Strang G. (2016) *Introduction to Linear Algebra*, 5th ed. Wellesley-Cambridge Press — chapter 2 (solving linear
  equations, elimination, inverses, $A = LU$) and §3.2–3.3 (null spaces, complete solution $\mathbf x_p + \mathbf x_n$).
- Isaak S., Manougian M. N. (1976) *Basic Concepts of Linear Algebra*. W. W. Norton — systems of linear equations,
  elementary operations and matrix inverses.
- Hoffman K., Kunze R. (1971) *Linear Algebra*, 2nd ed. Prentice-Hall — elementary matrices; invertible matrices as products
  of elementary matrices.
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — §2.2–2.3
  (systems of linear equations, Gaussian elimination, the “minus-1 trick” for null spaces).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences*. CRC Press — linear systems and matrices
  for scientists.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Newman A. M. *et al.* (2015) Robust enumeration of cell subsets from tissue expression profiles. *Nature Methods*
  12:453–457 (CIBERSORT, a deconvolution method).
- Orth J. D., Thiele I., Palsson B. Ø. (2010) What is flux balance analysis? *Nature Biotechnology* 28:245–248.

## ➡️ Next

**[Module 10 · 🌌 Vector spaces and subspaces](10_vector_spaces.ipynb)** — the null space of Section 8, the steady-state fluxes of Section 9 and the set of all
solutions of a homogeneous system share a property: you can add any two of them, or scale one, and stay in the set.
Module 10 gives that property a name — a **vector space**, or a **subspace** of a bigger one — and shows that the same
rules govern arrows, matrices, gene-expression profiles, growth curves and polynomials. It also explains why the solution
set of $A\mathbf x = \mathbf b$ with $\mathbf b\ne\mathbf 0$ (Section 7's plane in $\mathbb R^5$) is *not* a subspace.